In [3]:
# ================================================================
# CUSTOM XI PLAYER PERFORMANCE ML MODELS
# ================================================================
#
# PURPOSE
# -------
# Predict:
#
#   1. Runs scored by a batter against a bowler
#   2. Balls faced by a batter against a bowler
#   3. Dismissal probability
#
# The model uses historical batter-vs-bowler performance.
#
# IMPORTANT:
# Current-match results such as:
#
#   runs
#   balls
#   dismissals
#   strike rate
#   matchup score
#
# are NOT used as input features.
#
# Only PREVIOUS historical information is used.
#
# ================================================================

import os
import glob
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error


print("=" * 75)
print("BUILDING CUSTOM XI PLAYER PERFORMANCE MODELS")
print("=" * 75)


# ================================================================
# 1. FIND EXISTING FILES AUTOMATICALLY
# ================================================================

CURRENT_DIR = os.getcwd()

print()
print("Current directory:")
print(CURRENT_DIR)


matchup_path = None
profile_path = None


for root, dirs, files in os.walk(CURRENT_DIR):

    if "matchup_df.pkl" in files:

        matchup_path = os.path.join(
            root,
            "matchup_df.pkl"
        )

    if "player_profile.pkl" in files:

        profile_path = os.path.join(
            root,
            "player_profile.pkl"
        )


if matchup_path is None:

    raise FileNotFoundError(
        "matchup_df.pkl could not be found."
    )


if profile_path is None:

    raise FileNotFoundError(
        "player_profile.pkl could not be found."
    )


print()
print("Matchup file:")
print(matchup_path)

print()
print("Player profile:")
print(profile_path)


# ================================================================
# 2. LOAD DATA
# ================================================================

matchup_df = joblib.load(
    matchup_path
)

player_profile = joblib.load(
    profile_path
)


print()
print("=" * 75)
print("DATA LOADED")
print("=" * 75)

print(
    "Matchup shape:",
    matchup_df.shape
)

print(
    "Player profile shape:",
    player_profile.shape
)


# ================================================================
# 3. COPY DATA
# ================================================================

df = matchup_df.copy()

profile = player_profile.copy()


# ================================================================
# 4. NORMALIZE PLAYER NAMES
# ================================================================

df["batter"] = (
    df["batter"]
    .astype(str)
    .str.strip()
)

df["bowler"] = (
    df["bowler"]
    .astype(str)
    .str.strip()
)

profile["player"] = (
    profile["player"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 5. SORT HISTORICALLY
# ================================================================
#
# We want previous information to represent what was known
# before the current matchup.
#
# ================================================================

if "date" in df.columns:

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    df = df.sort_values(
        [
            "date",
            "match_id"
        ]
    )

else:

    df = df.sort_values(
        "match_id"
    )


# ================================================================
# 6. REMOVE INVALID RECORDS
# ================================================================

df = df[
    df["batter"].notna()
    &
    df["bowler"].notna()
]

df = df[
    (df["batter"] != "")
    &
    (df["bowler"] != "")
    &
    (df["batter"] != "nan")
    &
    (df["bowler"] != "nan")
]


# ================================================================
# 7. IMPORTANT:
#    USE ONLY PREVIOUS/HISTORICAL FEATURES
# ================================================================

HISTORICAL_FEATURES = [

    "previous_balls",
    "previous_runs",
    "previous_dismissals",
    "previous_fours",
    "previous_sixes",
    "previous_strike_rate",

    "last_5_runs",
    "last_5_balls",
    "last_5_strike_rate",
    "last_5_dismissals"
]


# Make sure missing columns are handled safely.

for col in HISTORICAL_FEATURES:

    if col not in df.columns:

        df[col] = 0


# ================================================================
# 8. CLEAN HISTORICAL FEATURES
# ================================================================

for col in HISTORICAL_FEATURES:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0)


# ================================================================
# 9. CREATE ADDITIONAL FEATURES
# ================================================================

df["previous_run_rate"] = np.where(

    df["previous_balls"] > 0,

    df["previous_runs"]
    /
    df["previous_balls"],

    0
)


df["last_5_run_rate"] = np.where(

    df["last_5_balls"] > 0,

    df["last_5_runs"]
    /
    df["last_5_balls"],

    0
)


df["previous_boundary_rate"] = np.where(

    df["previous_balls"] > 0,

    (
        df["previous_fours"]
        +
        df["previous_sixes"]
    )
    /
    df["previous_balls"],

    0
)


df["recent_boundary_rate"] = np.where(

    df["last_5_balls"] > 0,

    (
        df["last_5_runs"]
    )
    /
    df["last_5_balls"],

    0
)


df["dismissal_rate_history"] = np.where(

    df["previous_balls"] > 0,

    df["previous_dismissals"]
    /
    df["previous_balls"],

    0
)


# ================================================================
# 10. TARGET VARIABLES
# ================================================================
#
# Current matchup outcome becomes the target.
#
# FEATURES = what was known BEFORE the matchup
#
# TARGET = what actually happened
#
# ================================================================


# -------------------------------
# Runs
# -------------------------------

df["target_runs"] = pd.to_numeric(
    df["runs"],
    errors="coerce"
).fillna(0)


# -------------------------------
# Balls
# -------------------------------

df["target_balls"] = pd.to_numeric(
    df["balls"],
    errors="coerce"
).fillna(0)


# -------------------------------
# Dismissals
# -------------------------------

df["target_dismissals"] = pd.to_numeric(
    df["dismissals"],
    errors="coerce"
).fillna(0)


# ================================================================
# 11. REMOVE CURRENT-MATCH LEAKAGE
# ================================================================
#
# NEVER include:
#
#   runs
#   balls
#   dismissals
#   fours
#   sixes
#   matchup_sr
#   matchup_score
#
# as input features.
#
# ================================================================

MODEL_FEATURES = HISTORICAL_FEATURES + [

    "previous_run_rate",
    "last_5_run_rate",
    "previous_boundary_rate",
    "recent_boundary_rate",
    "dismissal_rate_history"
]


X = df[
    MODEL_FEATURES
].copy()


# ================================================================
# 12. CLEAN INF / NAN
# ================================================================

X = X.replace(
    [
        np.inf,
        -np.inf
    ],
    np.nan
)

X = X.fillna(0)


# ================================================================
# 13. TARGET: RUNS PER BALL
# ================================================================
#
# Predicting runs-per-ball is more useful than directly predicting
# total runs because the number of balls faced can vary.
#
# ================================================================

df["target_runs_per_ball"] = np.where(

    df["target_balls"] > 0,

    df["target_runs"]
    /
    df["target_balls"],

    0
)


y_runs = df[
    "target_runs_per_ball"
].astype(float)


# ================================================================
# 14. SPLIT DATA
# ================================================================

X_train_runs, X_test_runs, y_train_runs, y_test_runs = (
    train_test_split(
        X,
        y_runs,
        test_size=0.20,
        random_state=42
    )
)


# ================================================================
# 15. RUNS MODEL
# ================================================================

print()
print("=" * 75)
print("TRAINING RUNS MODEL")
print("=" * 75)


runs_model = RandomForestRegressor(

    n_estimators=500,

    max_depth=14,

    min_samples_leaf=4,

    max_features="sqrt",

    random_state=42,

    n_jobs=-1
)


runs_model.fit(
    X_train_runs,
    y_train_runs
)


runs_prediction = runs_model.predict(
    X_test_runs
)


runs_mae = mean_absolute_error(
    y_test_runs,
    runs_prediction
)


runs_rmse = np.sqrt(
    mean_squared_error(
        y_test_runs,
        runs_prediction
    )
)


print(
    "Runs-per-ball MAE:",
    round(
        runs_mae,
        5
    )
)

print(
    "Runs-per-ball RMSE:",
    round(
        runs_rmse,
        5
    )
)


# ================================================================
# 16. BALLS MODEL
# ================================================================
#
# Predict expected balls in the matchup.
#
# Log transformation prevents very large innings from dominating.
#
# ================================================================

print()
print("=" * 75)
print("TRAINING BALLS MODEL")
print("=" * 75)


df["target_log_balls"] = np.log1p(
    df["target_balls"]
)


y_balls = df[
    "target_log_balls"
].astype(float)


X_train_balls, X_test_balls, y_train_balls, y_test_balls = (
    train_test_split(
        X,
        y_balls,
        test_size=0.20,
        random_state=42
    )
)


balls_model = RandomForestRegressor(

    n_estimators=500,

    max_depth=14,

    min_samples_leaf=4,

    max_features="sqrt",

    random_state=42,

    n_jobs=-1
)


balls_model.fit(
    X_train_balls,
    y_train_balls
)


balls_prediction_log = balls_model.predict(
    X_test_balls
)


balls_prediction = np.expm1(
    balls_prediction_log
)

balls_actual = np.expm1(
    y_test_balls
)


balls_mae = mean_absolute_error(
    balls_actual,
    balls_prediction
)


print(
    "Balls MAE:",
    round(
        balls_mae,
        4
    )
)


# ================================================================
# 17. DISMISSAL MODEL
# ================================================================
#
# Instead of simply saying OUT/NOT OUT, predict dismissal
# probability.
#
# This allows different bowlers to produce different probabilities.
#
# ================================================================

print()
print("=" * 75)
print("TRAINING DISMISSAL PROBABILITY MODEL")
print("=" * 75)


# Dismissal probability per delivery

df["target_dismissal_rate"] = np.where(

    df["target_balls"] > 0,

    df["target_dismissals"]
    /
    df["target_balls"],

    0
)


y_dismissal = df[
    "target_dismissal_rate"
].astype(float)


X_train_dismissal, X_test_dismissal, y_train_dismissal, y_test_dismissal = (
    train_test_split(
        X,
        y_dismissal,
        test_size=0.20,
        random_state=42
    )
)


dismissal_model = RandomForestRegressor(

    n_estimators=500,

    max_depth=14,

    min_samples_leaf=4,

    max_features="sqrt",

    random_state=42,

    n_jobs=-1
)


dismissal_model.fit(
    X_train_dismissal,
    y_train_dismissal
)


dismissal_prediction = dismissal_model.predict(
    X_test_dismissal
)


dismissal_prediction = np.clip(
    dismissal_prediction,
    0,
    1
)


dismissal_mae = mean_absolute_error(
    y_test_dismissal,
    dismissal_prediction
)


print(
    "Dismissal-rate MAE:",
    round(
        dismissal_mae,
        6
    )
)


# ================================================================
# 18. BUILD BOWLER DISMISSAL HISTORY
# ================================================================
#
# This is used later to determine:
#
#   "Who is most likely to dismiss this batter?"
#
# We do NOT hard-code this.
#
# The frontend will compare:
#
#   Batter × Bowler 1
#   Batter × Bowler 2
#   Batter × Bowler 3
#   ...
#
# ================================================================

bowler_dismissal_history = (
    df
    .groupby(
        "bowler",
        as_index=False
    )
    .agg(
        total_dismissals=(
            "target_dismissals",
            "sum"
        ),

        total_balls=(
            "target_balls",
            "sum"
        )
    )
)


bowler_dismissal_history[
    "dismissal_rate"
] = np.where(

    bowler_dismissal_history[
        "total_balls"
    ] > 0,

    bowler_dismissal_history[
        "total_dismissals"
    ]
    /
    bowler_dismissal_history[
        "total_balls"
    ],

    0
)


# ================================================================
# 19. BUILD BATTER HISTORY
# ================================================================

batter_history = (
    df
    .groupby(
        "batter",
        as_index=False
    )
    .agg(

        historical_runs=(
            "target_runs",
            "sum"
        ),

        historical_balls=(
            "target_balls",
            "sum"
        ),

        historical_dismissals=(
            "target_dismissals",
            "sum"
        )
    )
)


batter_history[
    "historical_runs_per_ball"
] = np.where(

    batter_history[
        "historical_balls"
    ] > 0,

    batter_history[
        "historical_runs"
    ]
    /
    batter_history[
        "historical_balls"
    ],

    0
)


batter_history[
    "historical_dismissal_rate"
] = np.where(

    batter_history[
        "historical_balls"
    ] > 0,

    batter_history[
        "historical_dismissals"
    ]
    /
    batter_history[
        "historical_balls"
    ],

    0
)


# ================================================================
# 20. SAVE MODELS
# ================================================================

# Save into the SAME directory as this notebook.
# The frontend can later find them automatically.

SAVE_DIR = CURRENT_DIR


runs_model_path = os.path.join(
    SAVE_DIR,
    "custom_xi_runs_model.pkl"
)

balls_model_path = os.path.join(
    SAVE_DIR,
    "custom_xi_balls_model.pkl"
)

dismissal_model_path = os.path.join(
    SAVE_DIR,
    "custom_xi_dismissal_model.pkl"
)

features_path = os.path.join(
    SAVE_DIR,
    "custom_xi_model_features.pkl"
)

bowler_history_path = os.path.join(
    SAVE_DIR,
    "custom_xi_bowler_dismissal_history.pkl"
)

batter_history_path = os.path.join(
    SAVE_DIR,
    "custom_xi_batter_history.pkl"
)


joblib.dump(
    runs_model,
    runs_model_path
)

joblib.dump(
    balls_model,
    balls_model_path
)

joblib.dump(
    dismissal_model,
    dismissal_model_path
)

joblib.dump(
    MODEL_FEATURES,
    features_path
)

joblib.dump(
    bowler_dismissal_history,
    bowler_history_path
)

joblib.dump(
    batter_history,
    batter_history_path
)


# ================================================================
# 21. SAVE TRAINING DATA FOR PREDICTION ENGINE
# ================================================================

training_lookup_path = os.path.join(
    SAVE_DIR,
    "custom_xi_training_lookup.pkl"
)


training_lookup = df[
    [
        "batter",
        "bowler"
    ]
    +
    MODEL_FEATURES
].copy()


training_lookup.to_pickle(
    training_lookup_path
)


# ================================================================
# 22. FINAL REPORT
# ================================================================

print()
print("=" * 75)
print("CUSTOM XI PLAYER MODELS READY")
print("=" * 75)

print()

print(
    "Historical matchup records:",
    len(df)
)

print(
    "Unique batters:",
    df["batter"].nunique()
)

print(
    "Unique bowlers:",
    df["bowler"].nunique()
)

print(
    "Unique batter-bowler pairs:",
    df[
        ["batter", "bowler"]
    ]
    .drop_duplicates()
    .shape[0]
)

print()

print(
    "Runs model MAE:",
    round(
        runs_mae,
        5
    )
)

print(
    "Balls model MAE:",
    round(
        balls_mae,
        4
    )
)

print(
    "Dismissal model MAE:",
    round(
        dismissal_mae,
        6
    )
)

print()
print("=" * 75)
print("FILES CREATED")
print("=" * 75)

print(
    runs_model_path
)

print(
    balls_model_path
)

print(
    dismissal_model_path
)

print(
    features_path
)

print(
    bowler_history_path
)

print(
    batter_history_path
)

print(
    training_lookup_path
)

print()
print("=" * 75)
print("NEXT STEP: CUSTOM XI SCORECARD ENGINE")
print("=" * 75)

BUILDING CUSTOM XI PLAYER PERFORMANCE MODELS

Current directory:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks

Matchup file:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\notebooks\matchup_df.pkl

Player profile:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\notebooks\player_profile.pkl

DATA LOADED
Matchup shape: (61429, 22)
Player profile shape: (811, 13)

TRAINING RUNS MODEL
Runs-per-ball MAE: 0.69589
Runs-per-ball RMSE: 0.90835

TRAINING BALLS MODEL
Balls MAE: 2.345

TRAINING DISMISSAL PROBABILITY MODEL
Dismissal-rate MAE: 0.139728

CUSTOM XI PLAYER MODELS READY

Historical matchup records: 61429
Unique batters: 738
Unique bowlers: 577
Unique batter-bowler pairs: 31370

Runs model MAE: 0.69589
Balls model MAE: 2.345
Dismissal model MAE: 0.139728

FILES CREATED
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_runs_model.pkl
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_balls_model.pkl
c:\Use

In [4]:
# ================================================================
# CUSTOM XI PREDICTION ENGINE
# ================================================================
#
# Uses the trained ML models to simulate an innings.
#
# NO HARD-CODED:
#   runs
#   wickets
#   extras
#   dismissals
#   OUT players
#
# ================================================================

import os
import joblib
import numpy as np
import pandas as pd


print("=" * 75)
print("BUILDING CUSTOM XI SCORECARD ENGINE")
print("=" * 75)


# ================================================================
# 1. FIND TRAINED MODELS
# ================================================================

BASE_DIR = os.getcwd()


def find_file(filename):

    for root, dirs, files in os.walk(BASE_DIR):

        if filename in files:

            return os.path.join(
                root,
                filename
            )

    raise FileNotFoundError(
        f"{filename} was not found."
    )


runs_model_path = find_file(
    "custom_xi_runs_model.pkl"
)

balls_model_path = find_file(
    "custom_xi_balls_model.pkl"
)

dismissal_model_path = find_file(
    "custom_xi_dismissal_model.pkl"
)

features_path = find_file(
    "custom_xi_model_features.pkl"
)

training_lookup_path = find_file(
    "custom_xi_training_lookup.pkl"
)


runs_model = joblib.load(
    runs_model_path
)

balls_model = joblib.load(
    balls_model_path
)

dismissal_model = joblib.load(
    dismissal_model_path
)

MODEL_FEATURES = joblib.load(
    features_path
)

training_lookup = pd.read_pickle(
    training_lookup_path
)


print()
print("Models loaded successfully.")


# ================================================================
# 2. LOAD HISTORICAL MATCHUP DATA
# ================================================================

matchup_path = find_file(
    "matchup_df.pkl"
)

matchup_df = pd.read_pickle(
    matchup_path
)


matchup_df["batter"] = (
    matchup_df["batter"]
    .astype(str)
    .str.strip()
)

matchup_df["bowler"] = (
    matchup_df["bowler"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 3. BUILD FAST LOOKUPS
# ================================================================

pair_lookup = {}

for _, row in matchup_df.iterrows():

    key = (
        str(row["batter"]).strip(),
        str(row["bowler"]).strip()
    )

    pair_lookup[key] = row


print(
    "Historical matchup records:",
    len(pair_lookup)
)


# ================================================================
# 4. FEATURE CREATION FOR A BATTER × BOWLER
# ================================================================

def get_matchup_features(
    batter,
    bowler
):

    key = (
        str(batter).strip(),
        str(bowler).strip()
    )

    # ------------------------------------------------------------
    # Historical matchup exists
    # ------------------------------------------------------------

    if key in pair_lookup:

        row = pair_lookup[key]

        previous_balls = float(
            row.get(
                "previous_balls",
                0
            )
        )

        previous_runs = float(
            row.get(
                "previous_runs",
                0
            )
        )

        previous_dismissals = float(
            row.get(
                "previous_dismissals",
                0
            )
        )

        previous_fours = float(
            row.get(
                "previous_fours",
                0
            )
        )

        previous_sixes = float(
            row.get(
                "previous_sixes",
                0
            )
        )

        previous_sr = float(
            row.get(
                "previous_strike_rate",
                0
            )
        )

        last_5_runs = float(
            row.get(
                "last_5_runs",
                0
            )
        )

        last_5_balls = float(
            row.get(
                "last_5_balls",
                0
            )
        )

        last_5_sr = float(
            row.get(
                "last_5_strike_rate",
                0
            )
        )

        last_5_dismissals = float(
            row.get(
                "last_5_dismissals",
                0
            )
        )

    # ------------------------------------------------------------
    # No direct matchup
    # ------------------------------------------------------------

    else:

        # Search historical records for the batter.
        batter_rows = matchup_df[
            matchup_df["batter"]
            ==
            batter
        ]

        # Use batter's historical information as fallback.

        if len(batter_rows) > 0:

            latest = (
                batter_rows
                .sort_values(
                    "match_id"
                )
                .iloc[-1]
            )

            previous_balls = float(
                latest.get(
                    "previous_balls",
                    0
                )
            )

            previous_runs = float(
                latest.get(
                    "previous_runs",
                    0
                )
            )

            previous_dismissals = float(
                latest.get(
                    "previous_dismissals",
                    0
                )
            )

            previous_fours = float(
                latest.get(
                    "previous_fours",
                    0
                )
            )

            previous_sixes = float(
                latest.get(
                    "previous_sixes",
                    0
                )
            )

            previous_sr = float(
                latest.get(
                    "previous_strike_rate",
                    0
                )
            )

            last_5_runs = float(
                latest.get(
                    "last_5_runs",
                    0
                )
            )

            last_5_balls = float(
                latest.get(
                    "last_5_balls",
                    0
                )
            )

            last_5_sr = float(
                latest.get(
                    "last_5_strike_rate",
                    0
                )
            )

            last_5_dismissals = float(
                latest.get(
                    "last_5_dismissals",
                    0
                )
            )

        else:

            # Completely unseen batter.
            previous_balls = 0
            previous_runs = 0
            previous_dismissals = 0
            previous_fours = 0
            previous_sixes = 0
            previous_sr = 0
            last_5_runs = 0
            last_5_balls = 0
            last_5_sr = 0
            last_5_dismissals = 0


    # ------------------------------------------------------------
    # Derived features
    # ------------------------------------------------------------

    previous_run_rate = (

        previous_runs
        /
        previous_balls

        if previous_balls > 0

        else 0
    )


    last_5_run_rate = (

        last_5_runs
        /
        last_5_balls

        if last_5_balls > 0

        else 0
    )


    previous_boundary_rate = (

        (
            previous_fours
            +
            previous_sixes
        )
        /
        previous_balls

        if previous_balls > 0

        else 0
    )


    recent_boundary_rate = (

        last_5_runs
        /
        last_5_balls

        if last_5_balls > 0

        else 0
    )


    dismissal_rate_history = (

        previous_dismissals
        /
        previous_balls

        if previous_balls > 0

        else 0
    )


    # ------------------------------------------------------------
    # Create feature row
    # ------------------------------------------------------------

    values = {

        "previous_balls":
            previous_balls,

        "previous_runs":
            previous_runs,

        "previous_dismissals":
            previous_dismissals,

        "previous_fours":
            previous_fours,

        "previous_sixes":
            previous_sixes,

        "previous_strike_rate":
            previous_sr,

        "last_5_runs":
            last_5_runs,

        "last_5_balls":
            last_5_balls,

        "last_5_strike_rate":
            last_5_sr,

        "last_5_dismissals":
            last_5_dismissals,

        "previous_run_rate":
            previous_run_rate,

        "last_5_run_rate":
            last_5_run_rate,

        "previous_boundary_rate":
            previous_boundary_rate,

        "recent_boundary_rate":
            recent_boundary_rate,

        "dismissal_rate_history":
            dismissal_rate_history
    }


    X = pd.DataFrame(
        [values]
    )

    X = X[
        MODEL_FEATURES
    ]

    X = X.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )

    X = X.fillna(0)

    return X


# ================================================================
# 5. PREDICT ONE BATTER AGAINST ONE BOWLER
# ================================================================

def predict_batter_bowler(
    batter,
    bowler
):

    X = get_matchup_features(
        batter,
        bowler
    )


    # ------------------------------------------------------------
    # Runs per ball
    # ------------------------------------------------------------

    runs_per_ball = float(
        runs_model.predict(X)[0]
    )


    runs_per_ball = max(
        0,
        runs_per_ball
    )


    # ------------------------------------------------------------
    # Balls
    # ------------------------------------------------------------

    balls = float(
        np.expm1(
            balls_model.predict(X)[0]
        )
    )


    balls = max(
        1,
        balls
    )


    # ------------------------------------------------------------
    # Dismissal probability per ball
    # ------------------------------------------------------------

    dismissal_rate = float(
        dismissal_model.predict(X)[0]
    )


    dismissal_rate = np.clip(
        dismissal_rate,
        0,
        0.35
    )


    return {

        "runs_per_ball":
            runs_per_ball,

        "expected_balls":
            balls,

        "dismissal_rate":
            dismissal_rate
    }


# ================================================================
# 6. PREDICT BATTER AGAINST ALL OPPOSING BOWLERS
# ================================================================

def predict_batter_against_attack(
    batter,
    opposing_bowlers
):

    results = []

    for bowler in opposing_bowlers:

        prediction = predict_batter_bowler(
            batter,
            bowler
        )

        prediction[
            "batter"
        ] = batter

        prediction[
            "bowler"
        ] = bowler

        results.append(
            prediction
        )


    if len(results) == 0:

        return None


    result_df = pd.DataFrame(
        results
    )


    # ------------------------------------------------------------
    # Weighted average across opposing bowling attack
    # ------------------------------------------------------------

    expected_runs_per_ball = (
        result_df[
            "runs_per_ball"
        ]
        .mean()
    )


    expected_balls = (
        result_df[
            "expected_balls"
        ]
        .mean()
    )


    average_dismissal_rate = (
        result_df[
            "dismissal_rate"
        ]
        .mean()
    )


    return {

        "batter":
            batter,

        "expected_runs_per_ball":
            expected_runs_per_ball,

        "expected_balls":
            expected_balls,

        "dismissal_rate":
            average_dismissal_rate,

        "bowler_predictions":
            result_df
    }


# ================================================================
# 7. DETERMINE LIKELY DISMISSING BOWLER
# ================================================================

def choose_dismissing_bowler(
    batter,
    opposing_bowlers
):

    candidates = []


    for bowler in opposing_bowlers:

        prediction = predict_batter_bowler(
            batter,
            bowler
        )


        candidates.append({

            "bowler":
                bowler,

            "dismissal_rate":
                prediction[
                    "dismissal_rate"
                ]
        })


    if len(candidates) == 0:

        return None


    candidates_df = pd.DataFrame(
        candidates
    )


    # ------------------------------------------------------------
    # Use probability distribution instead of always selecting
    # the same bowler.
    #
    # Higher historical dismissal probability = higher chance.
    # ------------------------------------------------------------

    probabilities = (
        candidates_df[
            "dismissal_rate"
        ]
        .values
    )


    probabilities = (
        probabilities
        +
        0.001
    )


    probabilities = (
        probabilities
        /
        probabilities.sum()
    )


    selected_index = np.random.choice(
        len(candidates_df),
        p=probabilities
    )


    return candidates_df.iloc[
        selected_index
    ]["bowler"]


# ================================================================
# 8. SIMULATE ONE INNINGS
# ================================================================

def simulate_innings(
    batting_xi,
    bowling_attack,
    team_name="Team"
):

    print()
    print("=" * 75)
    print(
        f"SIMULATING {team_name}"
    )
    print("=" * 75)


    # ------------------------------------------------------------
    # Remove duplicate player names
    # ------------------------------------------------------------

    batting_xi = list(
        dict.fromkeys(
            batting_xi
        )
    )

    bowling_attack = list(
        dict.fromkeys(
            bowling_attack
        )
    )


    # ------------------------------------------------------------
    # Maximum 11 players
    # ------------------------------------------------------------

    batting_xi = batting_xi[
        :11
    ]

    bowling_attack = bowling_attack[
        :11
    ]


    if len(batting_xi) < 2:

        raise ValueError(
            "At least 2 batting players are required."
        )


    if len(bowling_attack) == 0:

        raise ValueError(
            "At least 1 bowler is required."
        )


    # ------------------------------------------------------------
    # Predict every batter against every bowler
    # ------------------------------------------------------------

    batter_predictions = {}


    for batter in batting_xi:

        result = predict_batter_against_attack(
            batter,
            bowling_attack
        )

        batter_predictions[
            batter
        ] = result


    # ------------------------------------------------------------
    # Calculate expected performance
    # ------------------------------------------------------------

    expected = []


    for batter in batting_xi:

        result = batter_predictions[
            batter
        ]


        expected_runs = (
            result[
                "expected_runs_per_ball"
            ]
            *
            result[
                "expected_balls"
            ]
        )


        expected.append({

            "batter":
                batter,

            "expected_runs":
                expected_runs,

            "expected_balls":
                result[
                    "expected_balls"
                ],

            "dismissal_rate":
                result[
                    "dismissal_rate"
                ]
        })


    expected_df = pd.DataFrame(
        expected
    )


    # ============================================================
    # 9. SIMULATE REALISTIC INDIVIDUAL SCORE
    # ============================================================

    scorecard = []


    for _, row in expected_df.iterrows():

        batter = row[
            "batter"
        ]


        expected_runs = max(
            0,
            row[
                "expected_runs"
            ]
        )


        expected_balls = max(
            1,
            row[
                "expected_balls"
            ]
        )


        dismissal_rate = np.clip(
            row[
                "dismissal_rate"
            ],
            0,
            0.35
        )


        # --------------------------------------------------------
        # Random variation around ML prediction
        # --------------------------------------------------------

        balls = int(
            np.random.normal(
                expected_balls,
                max(
                    2,
                    expected_balls * 0.25
                )
            )
        )


        balls = max(
            1,
            min(
                balls,
                60
            )
        )


        runs_per_ball = max(
            0,
            np.random.normal(
                expected_runs / expected_balls
                if expected_balls > 0
                else 0,

                0.35
            )
        )


        runs = int(
            round(
                runs_per_ball
                *
                balls
            )
        )


        # --------------------------------------------------------
        # Add realistic boundary variation
        # --------------------------------------------------------

        runs = max(
            0,
            runs
        )


        # --------------------------------------------------------
        # Dismissal
        # --------------------------------------------------------

        # Probability increases with balls faced.

        dismissal_probability = 1 - (
            (1 - dismissal_rate)
            ** balls
        )


        is_out = (
            np.random.random()
            <
            dismissal_probability
        )


        dismissed_by = None


        if is_out:

            dismissed_by = choose_dismissing_bowler(
                batter,
                bowling_attack
            )


        scorecard.append({

            "Batter":
                batter,

            "Runs":
                runs,

            "Balls":
                balls,

            "SR":
                round(
                    runs
                    /
                    balls
                    *
                    100,
                    1
                ),

            "Status":
                "OUT"
                if is_out
                else "NOT OUT",

            "Dismissed By":
                dismissed_by
                if is_out
                else ""
        })


    scorecard_df = pd.DataFrame(
        scorecard
    )


    # ============================================================
    # 10. REALISTIC BATTING ORDER
    # ============================================================

    # We now simulate wickets.
    #
    # If a player gets out, the next player comes in.
    #
    # Players after the last dismissal are marked DID NOT BAT.

    wickets = int(
        scorecard_df[
            "Status"
        ]
        .eq("OUT")
        .sum()
    )


    # Maximum 10 wickets.

    wickets = min(
        wickets,
        10
    )


    # Determine actual batters who faced balls.

    # Sort based on original XI order.

    actual_scorecard = []

    active_count = 0

    wickets_seen = 0


    for i, row in scorecard_df.iterrows():

        if active_count >= 11:

            break


        if wickets_seen >= 10:

            # Last surviving batter can remain not out.
            break


        actual_scorecard.append(
            row.to_dict()
        )


        active_count += 1


        if row["Status"] == "OUT":

            wickets_seen += 1


    # ------------------------------------------------------------
    # Players after final batting position
    # ------------------------------------------------------------

    actual_scorecard_df = pd.DataFrame(
        actual_scorecard
    )


    # ------------------------------------------------------------
    # Recalculate actual total
    # ------------------------------------------------------------

    if len(actual_scorecard_df) > 0:

        total_runs = int(
            actual_scorecard_df[
                "Runs"
            ].sum()
        )

        total_wickets = int(
            actual_scorecard_df[
                "Status"
            ]
            .eq("OUT")
            .sum()
        )

    else:

        total_runs = 0
        total_wickets = 0


    # ------------------------------------------------------------
    # Ensure maximum 10 wickets
    # ------------------------------------------------------------

    total_wickets = min(
        total_wickets,
        10
    )


    print()
    print(
        f"{team_name} predicted score:"
    )

    print(
        f"{total_runs}/{total_wickets}"
    )


    return {

        "scorecard":
            actual_scorecard_df,

        "runs":
            total_runs,

        "wickets":
            total_wickets,

        "batter_predictions":
            batter_predictions
    }


# ================================================================
# 11. TEST WITH CUSTOM XI
# ================================================================
#
# IMPORTANT:
#
# These are ONLY an example so we can test the engine.
#
# The final Streamlit application will get the players from
# YOUR selections.
#
# ================================================================

TEAM_1 = [
    "RG Sharma",
    "V Kohli",
    "Shubman Gill",
    "SS Iyer",
    "KL Rahul",
    "HH Pandya",
    "MS Dhoni"
]


TEAM_1_BOWLERS = [
    "JJ Bumrah",
    "Mohammed Siraj",
    "Kuldeep Yadav"
]


TEAM_2 = [
    "TM Head",
    "JC Buttler",
    "H Klaasen",
    "KA Pollard",
    "GJ Maxwell",
    "SO Hetmyer"
]


TEAM_2_BOWLERS = [
    "PJ Cummins",
    "MA Starc",
    "A Zampa",
    "K Rabada"
]


# ================================================================
# 12. SIMULATE BOTH INNINGS
# ================================================================

print()
print("=" * 75)
print("TESTING CUSTOM XI ENGINE")
print("=" * 75)


team1_result = simulate_innings(
    TEAM_1,
    TEAM_2_BOWLERS,
    "TEAM 1"
)


team2_result = simulate_innings(
    TEAM_2,
    TEAM_1_BOWLERS,
    "TEAM 2"
)


# ================================================================
# 13. DISPLAY SCORECARDS
# ================================================================

print()
print("=" * 75)
print("TEAM 1 SCORECARD")
print("=" * 75)

display(
    team1_result[
        "scorecard"
    ]
)


print()
print(
    "TOTAL:",
    f"{team1_result['runs']}/{team1_result['wickets']}"
)


print()
print("=" * 75)
print("TEAM 2 SCORECARD")
print("=" * 75)

display(
    team2_result[
        "scorecard"
    ]
)


print()
print(
    "TOTAL:",
    f"{team2_result['runs']}/{team2_result['wickets']}"
)


# ================================================================
# 14. WINNER
# ================================================================

print()
print("=" * 75)
print("PREDICTED RESULT")
print("=" * 75)


if (
    team1_result["runs"]
    >
    team2_result["runs"]
):

    print(
        "TEAM 1 predicted winner"
    )

elif (
    team2_result["runs"]
    >
    team1_result["runs"]
):

    print(
        "TEAM 2 predicted winner"
    )

else:

    print(
        "MATCH PREDICTED AS TIE"
    )


print()
print("=" * 75)
print("CUSTOM XI ENGINE READY")
print("=" * 75)

BUILDING CUSTOM XI SCORECARD ENGINE

Models loaded successfully.
Historical matchup records: 31370

TESTING CUSTOM XI ENGINE

SIMULATING TEAM 1

TEAM 1 predicted score:
36/0

SIMULATING TEAM 2

TEAM 2 predicted score:
41/2

TEAM 1 SCORECARD


,Batter,Runs,Balls,SR,Status,Dismissed By
0,RG Sharma,12,8,150.0,NOT OUT,
1,V Kohli,5,3,166.7,NOT OUT,
2,Shubman Gill,5,3,166.7,NOT OUT,
3,SS Iyer,3,2,150.0,NOT OUT,
4,KL Rahul,8,6,133.3,NOT OUT,
5,HH Pandya,1,1,100.0,NOT OUT,
6,MS Dhoni,2,3,66.7,NOT OUT,



TOTAL: 36/0

TEAM 2 SCORECARD


,Batter,Runs,Balls,SR,Status,Dismissed By
0,TM Head,10,6,166.7,NOT OUT,
1,JC Buttler,8,5,160.0,OUT,Kuldeep Yadav
2,H Klaasen,9,6,150.0,OUT,JJ Bumrah
3,KA Pollard,7,6,116.7,NOT OUT,
4,GJ Maxwell,4,3,133.3,NOT OUT,
5,SO Hetmyer,3,3,100.0,NOT OUT,



TOTAL: 41/2

PREDICTED RESULT
TEAM 2 predicted winner

CUSTOM XI ENGINE READY


In [5]:
# ================================================================
# PROPER BALL-BY-BALL CUSTOM XI ML SIMULATION ENGINE
# ================================================================
#
# IMPORTANT:
# This replaces the previous testing simulation.
#
# It does NOT hard-code:
#   - score
#   - wickets
#   - dismissals
#   - batter performances
#   - extras
#
# The selected batting XI and bowling XI are inputs.
#
# The ML models predict the outcome of each batter-vs-bowler
# interaction.
# ================================================================

import os
import numpy as np
import pandas as pd
import joblib


print("=" * 75)
print("BUILDING PROPER BALL-BY-BALL CUSTOM XI ML ENGINE")
print("=" * 75)


# ================================================================
# 1. FIND FILES
# ================================================================

BASE_DIR = os.getcwd()


def find_file(filename):

    # First search current directory and subdirectories

    for root, dirs, files in os.walk(BASE_DIR):

        if filename in files:

            return os.path.join(
                root,
                filename
            )

    raise FileNotFoundError(
        filename + " not found."
    )


runs_model = joblib.load(
    find_file(
        "custom_xi_runs_model.pkl"
    )
)

balls_model = joblib.load(
    find_file(
        "custom_xi_balls_model.pkl"
    )
)

dismissal_model = joblib.load(
    find_file(
        "custom_xi_dismissal_model.pkl"
    )
)

MODEL_FEATURES = joblib.load(
    find_file(
        "custom_xi_model_features.pkl"
    )
)

matchup_df = pd.read_pickle(
    find_file(
        "matchup_df.pkl"
    )
)


print()
print("ML models loaded.")
print(
    "Historical matchup rows:",
    len(matchup_df)
)


# ================================================================
# 2. NORMALIZE MATCHUP DATA
# ================================================================

matchup_df["batter"] = (
    matchup_df["batter"]
    .astype(str)
    .str.strip()
)

matchup_df["bowler"] = (
    matchup_df["bowler"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 3. SORT HISTORICALLY
# ================================================================

if "date" in matchup_df.columns:

    matchup_df["date"] = pd.to_datetime(
        matchup_df["date"],
        errors="coerce"
    )

    matchup_df = matchup_df.sort_values(
        [
            "date",
            "match_id"
        ]
    )


# ================================================================
# 4. BUILD MATCHUP LOOKUP
# ================================================================

pair_lookup = {}


for _, row in matchup_df.iterrows():

    key = (
        str(row["batter"]).strip(),
        str(row["bowler"]).strip()
    )

    pair_lookup[key] = row


# ================================================================
# 5. GET HISTORICAL MATCHUP
# ================================================================

def get_history(
    batter,
    bowler
):

    key = (
        str(batter).strip(),
        str(bowler).strip()
    )

    if key in pair_lookup:

        row = pair_lookup[key]

        return {

            "previous_balls":
                float(
                    row.get(
                        "previous_balls",
                        0
                    )
                ),

            "previous_runs":
                float(
                    row.get(
                        "previous_runs",
                        0
                    )
                ),

            "previous_dismissals":
                float(
                    row.get(
                        "previous_dismissals",
                        0
                    )
                ),

            "previous_fours":
                float(
                    row.get(
                        "previous_fours",
                        0
                    )
                ),

            "previous_sixes":
                float(
                    row.get(
                        "previous_sixes",
                        0
                    )
                ),

            "previous_strike_rate":
                float(
                    row.get(
                        "previous_strike_rate",
                        0
                    )
                ),

            "last_5_runs":
                float(
                    row.get(
                        "last_5_runs",
                        0
                    )
                ),

            "last_5_balls":
                float(
                    row.get(
                        "last_5_balls",
                        0
                    )
                ),

            "last_5_strike_rate":
                float(
                    row.get(
                        "last_5_strike_rate",
                        0
                    )
                ),

            "last_5_dismissals":
                float(
                    row.get(
                        "last_5_dismissals",
                        0
                    )
                )
        }


    # ------------------------------------------------------------
    # No direct matchup.
    #
    # Use the batter's historical records as fallback.
    # ------------------------------------------------------------

    batter_rows = matchup_df[
        matchup_df["batter"]
        ==
        str(batter).strip()
    ]


    if len(batter_rows) == 0:

        return {

            "previous_balls": 0,
            "previous_runs": 0,
            "previous_dismissals": 0,
            "previous_fours": 0,
            "previous_sixes": 0,
            "previous_strike_rate": 0,
            "last_5_runs": 0,
            "last_5_balls": 0,
            "last_5_strike_rate": 0,
            "last_5_dismissals": 0
        }


    row = batter_rows.iloc[-1]


    return {

        "previous_balls":
            float(
                row.get(
                    "previous_balls",
                    0
                )
            ),

        "previous_runs":
            float(
                row.get(
                    "previous_runs",
                    0
                )
            ),

        "previous_dismissals":
            float(
                row.get(
                    "previous_dismissals",
                    0
                )
            ),

        "previous_fours":
            float(
                row.get(
                    "previous_fours",
                    0
                )
            ),

        "previous_sixes":
            float(
                row.get(
                    "previous_sixes",
                    0
                )
            ),

        "previous_strike_rate":
            float(
                row.get(
                    "previous_strike_rate",
                    0
                )
            ),

        "last_5_runs":
            float(
                row.get(
                    "last_5_runs",
                    0
                )
            ),

        "last_5_balls":
            float(
                row.get(
                    "last_5_balls",
                    0
                )
            ),

        "last_5_strike_rate":
            float(
                row.get(
                    "last_5_strike_rate",
                    0
                )
            ),

        "last_5_dismissals":
            float(
                row.get(
                    "last_5_dismissals",
                    0
                )
            )
    }


# ================================================================
# 6. CREATE FEATURES
# ================================================================

def make_features(
    batter,
    bowler
):

    h = get_history(
        batter,
        bowler
    )


    previous_balls = h[
        "previous_balls"
    ]

    previous_runs = h[
        "previous_runs"
    ]

    previous_dismissals = h[
        "previous_dismissals"
    ]

    previous_fours = h[
        "previous_fours"
    ]

    previous_sixes = h[
        "previous_sixes"
    ]

    previous_sr = h[
        "previous_strike_rate"
    ]

    last_5_runs = h[
        "last_5_runs"
    ]

    last_5_balls = h[
        "last_5_balls"
    ]

    last_5_sr = h[
        "last_5_strike_rate"
    ]

    last_5_dismissals = h[
        "last_5_dismissals"
    ]


    previous_run_rate = (

        previous_runs
        /
        previous_balls

        if previous_balls > 0

        else 0
    )


    last_5_run_rate = (

        last_5_runs
        /
        last_5_balls

        if last_5_balls > 0

        else 0
    )


    previous_boundary_rate = (

        (
            previous_fours
            +
            previous_sixes
        )
        /
        previous_balls

        if previous_balls > 0

        else 0
    )


    recent_boundary_rate = (

        last_5_runs
        /
        last_5_balls

        if last_5_balls > 0

        else 0
    )


    dismissal_rate_history = (

        previous_dismissals
        /
        previous_balls

        if previous_balls > 0

        else 0
    )


    values = {

        "previous_balls":
            previous_balls,

        "previous_runs":
            previous_runs,

        "previous_dismissals":
            previous_dismissals,

        "previous_fours":
            previous_fours,

        "previous_sixes":
            previous_sixes,

        "previous_strike_rate":
            previous_sr,

        "last_5_runs":
            last_5_runs,

        "last_5_balls":
            last_5_balls,

        "last_5_strike_rate":
            last_5_sr,

        "last_5_dismissals":
            last_5_dismissals,

        "previous_run_rate":
            previous_run_rate,

        "last_5_run_rate":
            last_5_run_rate,

        "previous_boundary_rate":
            previous_boundary_rate,

        "recent_boundary_rate":
            recent_boundary_rate,

        "dismissal_rate_history":
            dismissal_rate_history
    }


    X = pd.DataFrame(
        [values]
    )


    X = X[
        MODEL_FEATURES
    ]


    X = X.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )


    X = X.fillna(0)


    return X


# ================================================================
# 7. PREDICT ONE BALL
# ================================================================

def predict_delivery(
    batter,
    bowler
):

    X = make_features(
        batter,
        bowler
    )


    # ------------------------------------------------------------
    # ML runs-per-ball prediction
    # ------------------------------------------------------------

    predicted_rpb = float(
        runs_model.predict(X)[0]
    )

    predicted_rpb = max(
        0,
        predicted_rpb
    )


    # ------------------------------------------------------------
    # ML dismissal-rate prediction
    # ------------------------------------------------------------

    dismissal_rate = float(
        dismissal_model.predict(X)[0]
    )

    dismissal_rate = np.clip(
        dismissal_rate,
        0,
        0.30
    )


    # ------------------------------------------------------------
    # Convert predicted run rate into probability distribution.
    #
    # We don't simply hard-code "0,1,2,4,6".
    #
    # The ML predicted run rate controls the distribution.
    # ------------------------------------------------------------

    expected = predicted_rpb


    # Probability weights around model prediction.

    possible_runs = np.array(
        [
            0,
            1,
            2,
            3,
            4,
            6
        ],
        dtype=int
    )


    # Distance from model expectation

    distances = np.abs(
        possible_runs
        -
        expected
    )


    weights = np.exp(
        -distances
        /
        0.9
    )


    weights = (
        weights
        /
        weights.sum()
    )


    predicted_runs = int(
        np.random.choice(
            possible_runs,
            p=weights
        )
    )


    # ------------------------------------------------------------
    # Dismissal
    # ------------------------------------------------------------

    wicket = (
        np.random.random()
        <
        dismissal_rate
    )


    return {

        "runs":
            predicted_runs,

        "wicket":
            bool(wicket),

        "dismissal_rate":
            dismissal_rate,

        "predicted_rpb":
            predicted_rpb
    }


# ================================================================
# 8. CHOOSE DISMISSING BOWLER
# ================================================================

def choose_dismissing_bowler(
    batter,
    bowling_attack
):

    probabilities = []


    for bowler in bowling_attack:

        X = make_features(
            batter,
            bowler
        )


        probability = float(
            dismissal_model.predict(X)[0]
        )


        probability = max(
            probability,
            0.0001
        )


        probabilities.append(
            probability
        )


    probabilities = np.array(
        probabilities
    )


    probabilities = (
        probabilities
        /
        probabilities.sum()
    )


    index = np.random.choice(
        len(bowling_attack),
        p=probabilities
    )


    return bowling_attack[
        index
    ]


# ================================================================
# 9. SIMULATE ONE INNINGS
# ================================================================

def simulate_innings(
    batting_xi,
    bowling_attack,
    overs=20,
    team_name="TEAM"
):

    batting_xi = list(
        dict.fromkeys(
            batting_xi
        )
    )

    bowling_attack = list(
        dict.fromkeys(
            bowling_attack
        )
    )


    if len(batting_xi) != 11:

        raise ValueError(
            "Batting XI must contain exactly 11 players."
        )


    if len(bowling_attack) < 1:

        raise ValueError(
            "At least one bowler is required."
        )


    # ============================================================
    # MATCH STATE
    # ============================================================

    total_runs = 0

    wickets = 0

    balls_bowled = 0

    max_balls = (
        overs
        *
        6
    )


    # ------------------------------------------------------------
    # Two active batsmen
    # ------------------------------------------------------------

    striker_index = 0

    non_striker_index = 1

    next_batter_index = 2


    striker = batting_xi[
        striker_index
    ]

    non_striker = batting_xi[
        non_striker_index
    ]


    # ------------------------------------------------------------
    # Individual stats
    # ------------------------------------------------------------

    stats = {}


    for player in batting_xi:

        stats[player] = {

            "runs": 0,

            "balls": 0,

            "status": "DID NOT BAT",

            "dismissed_by": ""
        }


    stats[striker]["status"] = (
        "NOT OUT"
    )

    stats[non_striker]["status"] = (
        "NOT OUT"
    )


    # ------------------------------------------------------------
    # Bowling stats
    # ------------------------------------------------------------

    bowling_stats = {}


    for bowler in bowling_attack:

        bowling_stats[bowler] = {

            "balls": 0,

            "runs": 0,

            "wickets": 0
        }


    # ============================================================
    # BALL-BY-BALL
    # ============================================================

    while (
        balls_bowled
        <
        max_balls
    ):

        # --------------------------------------------------------
        # ALL OUT
        # --------------------------------------------------------

        if wickets >= 10:

            break


        # --------------------------------------------------------
        # If no batter remains
        # --------------------------------------------------------

        if striker_index >= 11:

            break


        if non_striker_index >= 11:

            break


        # --------------------------------------------------------
        # Choose bowler dynamically
        # --------------------------------------------------------
        #
        # Every bowler can bowl.
        #
        # The final frontend can later enforce overs.
        #
        # For now choose based on historical dismissal strength.
        # --------------------------------------------------------

        bowler_scores = []


        for bowler in bowling_attack:

            X = make_features(
                striker,
                bowler
            )


            score = float(
                dismissal_model.predict(X)[0]
            )


            bowler_scores.append(
                max(
                    score,
                    0.001
                )
            )


        bowler_scores = np.array(
            bowler_scores
        )


        bowler_scores = (
            bowler_scores
            /
            bowler_scores.sum()
        )


        bowler_index = np.random.choice(
            len(bowling_attack),
            p=bowler_scores
        )


        bowler = bowling_attack[
            bowler_index
        ]


        # --------------------------------------------------------
        # Predict this delivery
        # --------------------------------------------------------

        outcome = predict_delivery(
            striker,
            bowler
        )


        runs = outcome[
            "runs"
        ]

        wicket = outcome[
            "wicket"
        ]


        # --------------------------------------------------------
        # Update batter
        # --------------------------------------------------------

        stats[striker][
            "runs"
        ] += runs

        stats[striker][
            "balls"
        ] += 1


        # --------------------------------------------------------
        # Update match
        # --------------------------------------------------------

        total_runs += runs

        balls_bowled += 1


        # --------------------------------------------------------
        # Update bowler
        # --------------------------------------------------------

        bowling_stats[
            bowler
        ]["balls"] += 1

        bowling_stats[
            bowler
        ]["runs"] += runs


        # ========================================================
        # WICKET
        # ========================================================

        if wicket:

            wickets += 1


            dismissed_by = bowler


            stats[striker][
                "status"
            ] = "OUT"


            stats[striker][
                "dismissed_by"
            ] = dismissed_by


            bowling_stats[
                bowler
            ]["wickets"] += 1


            # ----------------------------------------------------
            # New batter
            # ----------------------------------------------------

            if next_batter_index < 11:

                new_batter = batting_xi[
                    next_batter_index
                ]

                next_batter_index += 1


                stats[new_batter][
                    "status"
                ] = "NOT OUT"


                # New batter replaces striker

                striker_index = (
                    batting_xi.index(
                        striker
                    )
                )


                # If striker was dismissed,
                # new batter becomes striker.

                striker = new_batter

                striker_index = (
                    batting_xi.index(
                        striker
                    )
                )


            else:

                # No batter left

                break


        # ========================================================
        # NO WICKET
        # ========================================================

        else:

            # ----------------------------------------------------
            # Strike changes according to odd runs.
            # ----------------------------------------------------

            if runs % 2 == 1:

                (
                    striker,
                    non_striker
                ) = (
                    non_striker,
                    striker
                )


        # --------------------------------------------------------
        # End of over
        # --------------------------------------------------------

        if balls_bowled % 6 == 0:

            (
                striker,
                non_striker
            ) = (
                non_striker,
                striker
            )


    # ============================================================
    # FINAL BATTER STATUSES
    # ============================================================

    for player in batting_xi:

        if stats[player]["balls"] > 0:

            if stats[player]["status"] != "OUT":

                stats[player][
                    "status"
                ] = "NOT OUT"

        else:

            stats[player][
                "status"
            ] = "DID NOT BAT"


    # ============================================================
    # SCORECARD
    # ============================================================

    scorecard = []


    for position, player in enumerate(
        batting_xi,
        start=1
    ):

        item = stats[player]


        runs = item[
            "runs"
        ]

        balls = item[
            "balls"
        ]


        sr = (

            runs
            /
            balls
            *
            100

            if balls > 0

            else 0
        )


        scorecard.append({

            "No":
                position,

            "Batter":
                player,

            "Runs":
                runs,

            "Balls":
                balls,

            "SR":
                round(
                    sr,
                    1
                ),

            "Status":
                item[
                    "status"
                ],

            "Dismissed By":
                item[
                    "dismissed_by"
                ]
        })


    scorecard_df = pd.DataFrame(
        scorecard
    )


    # ============================================================
    # BOWLING CARD
    # ============================================================

    bowling_card = []


    for bowler in bowling_attack:

        item = bowling_stats[
            bowler
        ]


        bowling_card.append({

            "Bowler":
                bowler,

            "Balls":
                item["balls"],

            "Runs":
                item["runs"],

            "Wickets":
                item["wickets"],

            "Overs":
                f"{item['balls'] // 6}."
                f"{item['balls'] % 6}"
        })


    bowling_df = pd.DataFrame(
        bowling_card
    )


    # ============================================================
    # FINAL RESULT
    # ============================================================

    result = {

        "team":
            team_name,

        "runs":
            int(total_runs),

        "wickets":
            int(wickets),

        "balls":
            int(balls_bowled),

        "overs":
            f"{balls_bowled // 6}."
            f"{balls_bowled % 6}",

        "scorecard":
            scorecard_df,

        "bowling":
            bowling_df
    }


    return result


print()
print("=" * 75)
print("PROPER ML SCORECARD ENGINE READY")
print("=" * 75)

BUILDING PROPER BALL-BY-BALL CUSTOM XI ML ENGINE

ML models loaded.
Historical matchup rows: 61429

PROPER ML SCORECARD ENGINE READY


In [1]:
import os

print("=" * 70)
print("MODEL FILES")
print("=" * 70)

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith((".pkl", ".joblib")):
            print(os.path.join(root, file))

MODEL FILES
.\custom_xi_balls_model.pkl
.\custom_xi_batter_history.pkl
.\custom_xi_bowler_dismissal_history.pkl
.\custom_xi_dismissal_model.pkl
.\custom_xi_model_features.pkl
.\custom_xi_runs_model.pkl
.\custom_xi_training_lookup.pkl
.\final_feature_columns.pkl
.\final_ipl_match_predictor.pkl
.\final_prediction_threshold.pkl
.\versus_logistic_regression.pkl
.\notebooks\matchup_df.pkl
.\notebooks\player_profile.pkl


In [2]:
# ================================================================
# PROPER STOCHASTIC BALL-BY-BALL CUSTOM XI ML ENGINE
# ================================================================
#
# PURPOSE
# -------
# This engine simulates a complete T20 innings using the trained
# Custom XI ML models.
#
# IMPORTANT:
#
# NO hard-coded:
#   - player performances
#   - player runs
#   - player wickets
#   - dismissal relationships
#   - winner
#   - score
#
# The only fixed values are CRICKET RULES:
#   - 6 legal balls = 1 over
#   - maximum 4 overs per bowler in a 20-over innings
#   - 10 wickets ends an innings
#   - normal batting runs are selected from legal cricket outcomes
#
# The actual probabilities are generated from the ML models and
# historical matchup data.
#
# ================================================================


import os
import numpy as np
import pandas as pd
import joblib

from collections import defaultdict


print("=" * 80)
print("PROPER STOCHASTIC BALL-BY-BALL CUSTOM XI ML ENGINE")
print("=" * 80)


# ================================================================
# 1. FIND PROJECT DIRECTORY
# ================================================================

BASE_DIR = os.getcwd()

print()
print("Current directory:")
print(BASE_DIR)


# ================================================================
# 2. FILE FINDER
# ================================================================

def find_file(filename):

    # Search current directory first
    direct_path = os.path.join(
        BASE_DIR,
        filename
    )

    if os.path.isfile(direct_path):
        return direct_path

    # Search recursively
    for root, dirs, files in os.walk(BASE_DIR):

        if filename in files:

            return os.path.join(
                root,
                filename
            )

    raise FileNotFoundError(
        f"{filename} was not found under {BASE_DIR}"
    )


# ================================================================
# 3. LOAD ML MODELS
# ================================================================

print()
print("=" * 80)
print("LOADING ML MODELS")
print("=" * 80)


RUNS_MODEL_PATH = find_file(
    "custom_xi_runs_model.pkl"
)

BALLS_MODEL_PATH = find_file(
    "custom_xi_balls_model.pkl"
)

DISMISSAL_MODEL_PATH = find_file(
    "custom_xi_dismissal_model.pkl"
)

FEATURES_PATH = find_file(
    "custom_xi_model_features.pkl"
)

MATCHUP_PATH = find_file(
    "matchup_df.pkl"
)


runs_model = joblib.load(
    RUNS_MODEL_PATH
)

balls_model = joblib.load(
    BALLS_MODEL_PATH
)

dismissal_model = joblib.load(
    DISMISSAL_MODEL_PATH
)

MODEL_FEATURES = joblib.load(
    FEATURES_PATH
)

matchup_df = pd.read_pickle(
    MATCHUP_PATH
)


print()
print("Runs model loaded:")
print(RUNS_MODEL_PATH)

print()
print("Balls model loaded:")
print(BALLS_MODEL_PATH)

print()
print("Dismissal model loaded:")
print(DISMISSAL_MODEL_PATH)

print()
print("Feature file loaded:")
print(FEATURES_PATH)

print()
print("Historical matchup file loaded:")
print(MATCHUP_PATH)

print()
print("Historical matchup rows:")
print(len(matchup_df))


# ================================================================
# 4. NORMALIZE HISTORICAL DATA
# ================================================================

matchup_df = matchup_df.copy()


required_basic_columns = [
    "batter",
    "bowler"
]


for column in required_basic_columns:

    if column not in matchup_df.columns:

        raise ValueError(
            f"Required column '{column}' "
            f"is missing from matchup_df."
        )


matchup_df["batter"] = (
    matchup_df["batter"]
    .astype(str)
    .str.strip()
)

matchup_df["bowler"] = (
    matchup_df["bowler"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 5. NUMERIC COLUMNS
# ================================================================

numeric_columns = [

    "balls",
    "runs",
    "fours",
    "sixes",
    "dismissals",

    "previous_balls",
    "previous_runs",
    "previous_dismissals",
    "previous_fours",
    "previous_sixes",
    "previous_strike_rate",

    "last_5_runs",
    "last_5_balls",
    "last_5_strike_rate",
    "last_5_dismissals",

    "matchup_sr",
    "matchup_score"
]


for column in numeric_columns:

    if column in matchup_df.columns:

        matchup_df[column] = pd.to_numeric(
            matchup_df[column],
            errors="coerce"
        ).fillna(0)


# ================================================================
# 6. REMOVE INVALID NAMES
# ================================================================

matchup_df = matchup_df[
    matchup_df["batter"].notna()
    &
    matchup_df["bowler"].notna()
]


matchup_df = matchup_df[
    (matchup_df["batter"] != "")
    &
    (matchup_df["bowler"] != "")
    &
    (matchup_df["batter"] != "nan")
    &
    (matchup_df["bowler"] != "nan")
].copy()


# ================================================================
# 7. SORT HISTORICAL DATA
# ================================================================

if "date" in matchup_df.columns:

    matchup_df["date"] = pd.to_datetime(
        matchup_df["date"],
        errors="coerce"
    )


sort_columns = []

if "date" in matchup_df.columns:
    sort_columns.append("date")

if "match_id" in matchup_df.columns:
    sort_columns.append("match_id")


if len(sort_columns) > 0:

    matchup_df = matchup_df.sort_values(
        sort_columns
    )


matchup_df = matchup_df.reset_index(
    drop=True
)


# ================================================================
# 8. BUILD AGGREGATED HISTORICAL LOOKUPS
# ================================================================
#
# IMPORTANT:
# We do NOT do:
#
#     pair_lookup[key] = row
#
# because that keeps only one historical row.
#
# Instead we aggregate ALL historical records for a matchup.
#
# ================================================================


PAIR_COLUMNS = [
    "balls",
    "runs",
    "fours",
    "sixes",
    "dismissals",

    "previous_balls",
    "previous_runs",
    "previous_dismissals",
    "previous_fours",
    "previous_sixes",
    "previous_strike_rate",

    "last_5_runs",
    "last_5_balls",
    "last_5_strike_rate",
    "last_5_dismissals",

    "matchup_sr",
    "matchup_score"
]


available_pair_columns = [
    c
    for c in PAIR_COLUMNS
    if c in matchup_df.columns
]


pair_history = (
    matchup_df
    .groupby(
        [
            "batter",
            "bowler"
        ],
        as_index=False
    )[available_pair_columns]
    .mean()
)


pair_counts = (
    matchup_df
    .groupby(
        [
            "batter",
            "bowler"
    ])
    .size()
    .reset_index(
        name="history_rows"
    )
)


pair_history = pair_history.merge(
    pair_counts,
    on=[
        "batter",
        "bowler"
    ],
    how="left"
)


# ================================================================
# 9. BATTER AGGREGATE HISTORY
# ================================================================

batter_group = (
    matchup_df
    .groupby(
        "batter",
        as_index=False
    )
)


batter_history = batter_group.agg(

    batter_total_runs=(
        "runs",
        "sum"
    ),

    batter_total_balls=(
        "balls",
        "sum"
    ),

    batter_total_dismissals=(
        "dismissals",
        "sum"
    ),

    batter_avg_runs=(
        "runs",
        "mean"
    ),

    batter_avg_balls=(
        "balls",
        "mean"
    ),

    batter_avg_dismissals=(
        "dismissals",
        "mean"
    ),

    batter_avg_strike_rate=(
        "matchup_sr",
        "mean"
    )
)


# ================================================================
# 10. BOWLER AGGREGATE HISTORY
# ================================================================

bowler_group = (
    matchup_df
    .groupby(
        "bowler",
        as_index=False
    )
)


bowler_history = bowler_group.agg(

    bowler_total_runs=(
        "runs",
        "sum"
    ),

    bowler_total_balls=(
        "balls",
        "sum"
    ),

    bowler_total_dismissals=(
        "dismissals",
        "sum"
    ),

    bowler_avg_runs=(
        "runs",
        "mean"
    ),

    bowler_avg_balls=(
        "balls",
        "mean"
    ),

    bowler_avg_dismissals=(
        "dismissals",
        "mean"
    )
)


# ================================================================
# 11. FAST LOOKUPS
# ================================================================

pair_lookup = pair_history.set_index(
    [
        "batter",
        "bowler"
    ]
)

batter_lookup = batter_history.set_index(
    "batter"
)

bowler_lookup = bowler_history.set_index(
    "bowler"
)


# ================================================================
# 12. DEFAULT HISTORY
# ================================================================

def empty_history():

    return {

        "previous_balls": 0.0,
        "previous_runs": 0.0,
        "previous_dismissals": 0.0,
        "previous_fours": 0.0,
        "previous_sixes": 0.0,
        "previous_strike_rate": 0.0,

        "last_5_runs": 0.0,
        "last_5_balls": 0.0,
        "last_5_strike_rate": 0.0,
        "last_5_dismissals": 0.0,

        "matchup_sr": 0.0,
        "matchup_score": 0.0,

        "balls": 0.0,
        "runs": 0.0,
        "dismissals": 0.0,

        "history_rows": 0.0
    }


# ================================================================
# 13. GET COMPLETE MATCHUP HISTORY
# ================================================================

def get_matchup_history(
    batter,
    bowler
):

    batter = str(batter).strip()
    bowler = str(bowler).strip()


    result = empty_history()


    # ------------------------------------------------------------
    # Direct batter-vs-bowler history
    # ------------------------------------------------------------

    key = (
        batter,
        bowler
    )


    if key in pair_lookup.index:

        row = pair_lookup.loc[key]


        # If duplicate MultiIndex somehow exists
        if isinstance(row, pd.DataFrame):

            row = row.iloc[-1]


        for column in result.keys():

            if column in row.index:

                value = row[column]

                if pd.notna(value):

                    try:

                        result[column] = float(
                            value
                        )

                    except Exception:

                        pass


    # ------------------------------------------------------------
    # Fallback: batter historical data
    # ------------------------------------------------------------

    else:

        if batter in batter_lookup.index:

            row = batter_lookup.loc[
                batter
            ]

            if isinstance(row, pd.DataFrame):

                row = row.iloc[-1]


            if "batter_total_balls" in row:

                result["previous_balls"] = float(
                    row["batter_total_balls"]
                )

            if "batter_total_runs" in row:

                result["previous_runs"] = float(
                    row["batter_total_runs"]
                )

            if "batter_total_dismissals" in row:

                result["previous_dismissals"] = float(
                    row["batter_total_dismissals"]
                )

            if "batter_avg_strike_rate" in row:

                result["previous_strike_rate"] = float(
                    row["batter_avg_strike_rate"]
                )


    return result


# ================================================================
# 14. BUILD MODEL FEATURE VECTOR
# ================================================================
#
# The important part here is:
#
# We construct the input according to the feature names saved
# with the trained model.
#
# This prevents accidental feature-order problems.
#
# ================================================================


def make_features(
    batter,
    bowler,
    current_match_state=None
):

    h = get_matchup_history(
        batter,
        bowler
    )


    # ------------------------------------------------------------
    # Current simulation state
    # ------------------------------------------------------------

    if current_match_state is None:

        current_match_state = {}


    current_batter_runs = float(
        current_match_state.get(
            "batter_runs",
            0
        )
    )

    current_batter_balls = float(
        current_match_state.get(
            "batter_balls",
            0
        )
    )

    current_innings_balls = float(
        current_match_state.get(
            "innings_balls",
            0
        )
    )

    current_wickets = float(
        current_match_state.get(
            "wickets",
            0
        )
    )


    # ------------------------------------------------------------
    # Derived values
    # ------------------------------------------------------------

    previous_balls = h[
        "previous_balls"
    ]

    previous_runs = h[
        "previous_runs"
    ]

    previous_dismissals = h[
        "previous_dismissals"
    ]

    previous_fours = h[
        "previous_fours"
    ]

    previous_sixes = h[
        "previous_sixes"
    ]

    previous_sr = h[
        "previous_strike_rate"
    ]


    last_5_runs = h[
        "last_5_runs"
    ]

    last_5_balls = h[
        "last_5_balls"
    ]

    last_5_sr = h[
        "last_5_strike_rate"
    ]

    last_5_dismissals = h[
        "last_5_dismissals"
    ]


    previous_run_rate = (

        previous_runs
        /
        previous_balls

        if previous_balls > 0

        else 0.0
    )


    last_5_run_rate = (

        last_5_runs
        /
        last_5_balls

        if last_5_balls > 0

        else 0.0
    )


    previous_boundary_rate = (

        (
            previous_fours
            +
            previous_sixes
        )
        /
        previous_balls

        if previous_balls > 0

        else 0.0
    )


    recent_boundary_rate = (

        (
            last_5_runs
            /
            last_5_balls
        )

        if last_5_balls > 0

        else 0.0
    )


    dismissal_rate_history = (

        previous_dismissals
        /
        previous_balls

        if previous_balls > 0

        else 0.0
    )


    # ------------------------------------------------------------
    # Feature dictionary
    # ------------------------------------------------------------

    values = {

        # Direct matchup
        "balls":
            h["balls"],

        "runs":
            h["runs"],

        "dismissals":
            h["dismissals"],

        "innings":
            h["history_rows"],

        "strike_rate":
            h["matchup_sr"],

        "runs_per_ball":
            (
                h["runs"]
                /
                h["balls"]
                if h["balls"] > 0
                else 0.0
            ),

        "dismissal_rate":
            (
                h["dismissals"]
                /
                h["balls"]
                if h["balls"] > 0
                else 0.0
            ),


        # Batter history
        "total_runs":
            (
                batter_lookup.loc[
                    batter,
                    "batter_total_runs"
                ]
                if batter in batter_lookup.index
                else 0.0
            ),

        "total_balls":
            (
                batter_lookup.loc[
                    batter,
                    "batter_total_balls"
                ]
                if batter in batter_lookup.index
                else 0.0
            ),

        "total_dismissals":
            (
                batter_lookup.loc[
                    batter,
                    "batter_total_dismissals"
                ]
                if batter in batter_lookup.index
                else 0.0
            ),

        "matches":
            h["history_rows"],

        "batting_sr":
            (
                (
                    batter_lookup.loc[
                        batter,
                        "batter_total_runs"
                    ]
                    /
                    batter_lookup.loc[
                        batter,
                        "batter_total_balls"
                    ]
                    *
                    100
                )
                if (
                    batter in batter_lookup.index
                    and
                    float(
                        batter_lookup.loc[
                            batter,
                            "batter_total_balls"
                        ]
                    ) > 0
                )
                else 0.0
            ),

        "runs_per_match":
            (
                batter_lookup.loc[
                    batter,
                    "batter_avg_runs"
                ]
                if batter in batter_lookup.index
                else 0.0
            ),

        "balls_per_match":
            (
                batter_lookup.loc[
                    batter,
                    "batter_avg_balls"
                ]
                if batter in batter_lookup.index
                else 0.0
            ),

        "overall_dismissal_rate":
            (
                (
                    batter_lookup.loc[
                        batter,
                        "batter_total_dismissals"
                    ]
                    /
                    batter_lookup.loc[
                        batter,
                        "batter_total_balls"
                    ]
                )
                if (
                    batter in batter_lookup.index
                    and
                    float(
                        batter_lookup.loc[
                            batter,
                            "batter_total_balls"
                        ]
                    ) > 0
                )
                else 0.0
            ),


        # Bowler history
        "bowling_runs_conceded":
            (
                bowler_lookup.loc[
                    bowler,
                    "bowler_total_runs"
                ]
                if bowler in bowler_lookup.index
                else 0.0
            ),

        "bowling_balls":
            (
                bowler_lookup.loc[
                    bowler,
                    "bowler_total_balls"
                ]
                if bowler in bowler_lookup.index
                else 0.0
            ),

        "bowling_wickets":
            (
                bowler_lookup.loc[
                    bowler,
                    "bowler_total_dismissals"
                ]
                if bowler in bowler_lookup.index
                else 0.0
            ),

        "bowling_economy":
            (
                (
                    bowler_lookup.loc[
                        bowler,
                        "bowler_total_runs"
                    ]
                    /
                    bowler_lookup.loc[
                        bowler,
                        "bowler_total_balls"
                    ]
                    *
                    6
                )
                if (
                    bowler in bowler_lookup.index
                    and
                    float(
                        bowler_lookup.loc[
                            bowler,
                            "bowler_total_balls"
                        ]
                    ) > 0
                )
                else 0.0
            ),

        "bowler_wicket_rate":
            (
                (
                    bowler_lookup.loc[
                        bowler,
                        "bowler_total_dismissals"
                    ]
                    /
                    bowler_lookup.loc[
                        bowler,
                        "bowler_total_balls"
                    ]
                )
                if (
                    bowler in bowler_lookup.index
                    and
                    float(
                        bowler_lookup.loc[
                            bowler,
                            "bowler_total_balls"
                        ]
                    ) > 0
                )
                else 0.0
            ),


        # Interaction
        "matchup_sr_difference":
            (
                h["matchup_sr"]
                -
                (
                    batter_lookup.loc[
                        batter,
                        "batter_avg_strike_rate"
                    ]
                    if batter in batter_lookup.index
                    else 0.0
                )
            ),

        "matchup_runs_per_ball_difference":
            (
                (
                    h["runs"]
                    /
                    h["balls"]
                    if h["balls"] > 0
                    else 0.0
                )
                -
                (
                    (
                        batter_lookup.loc[
                            batter,
                            "batter_total_runs"
                        ]
                        /
                        batter_lookup.loc[
                            batter,
                            "batter_total_balls"
                        ]
                    )
                    if (
                        batter in batter_lookup.index
                        and
                        float(
                            batter_lookup.loc[
                                batter,
                                "batter_total_balls"
                            ]
                        ) > 0
                    )
                    else 0.0
                )
            ),

        "matchup_dismissal_difference":
            (
                (
                    h["dismissals"]
                    /
                    h["balls"]
                    if h["balls"] > 0
                    else 0.0
                )
                -
                (
                    (
                        batter_lookup.loc[
                            batter,
                            "batter_total_dismissals"
                        ]
                        /
                        batter_lookup.loc[
                            batter,
                            "batter_total_balls"
                        ]
                    )
                    if (
                        batter in batter_lookup.index
                        and
                        float(
                            batter_lookup.loc[
                                batter,
                                "batter_total_balls"
                            ]
                        ) > 0
                    )
                    else 0.0
                )
            ),

        "bowler_pressure":
            (
                (
                    bowler_lookup.loc[
                        bowler,
                        "bowler_total_dismissals"
                    ]
                    /
                    bowler_lookup.loc[
                        bowler,
                        "bowler_total_balls"
                    ]
                )
                if (
                    bowler in bowler_lookup.index
                    and
                    float(
                        bowler_lookup.loc[
                            bowler,
                            "bowler_total_balls"
                        ]
                    ) > 0
                )
                else 0.0
            ),

        "matchup_experience":
            np.log1p(
                h["balls"]
            ),

        "matchup_innings_log":
            np.log1p(
                h["history_rows"]
            ),


        # Historical features
        "previous_balls":
            previous_balls,

        "previous_runs":
            previous_runs,

        "previous_dismissals":
            previous_dismissals,

        "previous_fours":
            previous_fours,

        "previous_sixes":
            previous_sixes,

        "previous_strike_rate":
            previous_sr,

        "last_5_runs":
            last_5_runs,

        "last_5_balls":
            last_5_balls,

        "last_5_strike_rate":
            last_5_sr,

        "last_5_dismissals":
            last_5_dismissals,

        "previous_run_rate":
            previous_run_rate,

        "last_5_run_rate":
            last_5_run_rate,

        "previous_boundary_rate":
            previous_boundary_rate,

        "recent_boundary_rate":
            recent_boundary_rate,

        "dismissal_rate_history":
            dismissal_rate_history,


        # Current simulation state
        "current_batter_runs":
            current_batter_runs,

        "current_batter_balls":
            current_batter_balls,

        "current_innings_balls":
            current_innings_balls,

        "current_wickets":
            current_wickets
    }


    # ------------------------------------------------------------
    # Build dataframe
    # ------------------------------------------------------------

    row = {}

    for feature in MODEL_FEATURES:

        row[feature] = values.get(
            feature,
            0.0
        )


    X = pd.DataFrame(
        [row],
        columns=MODEL_FEATURES
    )


    X = X.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )


    X = X.fillna(0.0)


    return X


# ================================================================
# 15. SAFE MODEL PREDICTION
# ================================================================

def get_runs_prediction(
    batter,
    bowler,
    state
):

    X = make_features(
        batter,
        bowler,
        state
    )


    prediction = float(
        runs_model.predict(X)[0]
    )


    if not np.isfinite(prediction):

        prediction = 0.0


    # Runs per ball cannot be negative.
    prediction = max(
        0.0,
        prediction
    )


    return prediction


# ================================================================
# 16. DISMISSAL PROBABILITY
# ================================================================

def get_dismissal_probability(
    batter,
    bowler,
    state
):

    X = make_features(
        batter,
        bowler,
        state
    )


    probabilities = dismissal_model.predict_proba(
        X
    )[0]


    # Find class 1 if available
    classes = list(
        dismissal_model.classes_
    )


    if 1 in classes:

        index = classes.index(1)

        probability = float(
            probabilities[index]
        )

    else:

        probability = 0.0


    if not np.isfinite(probability):

        probability = 0.0


    return float(
        np.clip(
            probability,
            0.0,
            0.95
        )
    )


# ================================================================
# 17. STOCHASTIC RUN DISTRIBUTION
# ================================================================
#
# We do NOT say:
#
#   Rohit = 40 runs
#   Kohli = 50 runs
#
# etc.
#
# Instead the ML model gives expected runs per ball.
#
# That expectation is converted into a probability distribution
# and sampled.
#
# Therefore:
#
# Same batter + same bowler
# can produce different outcomes.
#
# ================================================================


def run_probabilities(
    expected_runs_per_ball
):

    expected = float(
        expected_runs_per_ball
    )


    if not np.isfinite(expected):

        expected = 0.0


    expected = max(
        0.0,
        expected
    )


    # ------------------------------------------------------------
    # Legal batter scoring outcomes.
    #
    # These are cricket outcome categories, NOT player-specific
    # hard-coded predictions.
    # ------------------------------------------------------------

    outcomes = np.array(
        [
            0,
            1,
            2,
            3,
            4,
            6
        ],
        dtype=int
    )


    # ------------------------------------------------------------
    # Build a smooth probability distribution centered around
    # the ML expected value.
    # ------------------------------------------------------------

    scale = max(
        0.65,
        0.85 * (1.0 + expected)
    )


    distance = (
        outcomes
        -
        expected
    )


    weights = np.exp(
        -
        (
            distance ** 2
        )
        /
        (
            2.0 * scale ** 2
        )
    )


    # ------------------------------------------------------------
    # Slightly favor 0/1/2 for very low expected values and
    # boundaries when model expectation is high.
    #
    # The magnitude is derived from the model expectation.
    # ------------------------------------------------------------

    boundary_strength = np.clip(
        expected / 1.5,
        0.0,
        1.0
    )


    boundary_multiplier = np.ones(
        len(outcomes)
    )


    for i, value in enumerate(
        outcomes
    ):

        if value in [
            4,
            6
        ]:

            boundary_multiplier[i] += (
                0.35
                *
                boundary_strength
            )


    weights *= boundary_multiplier


    # ------------------------------------------------------------
    # Normalize
    # ------------------------------------------------------------

    total = weights.sum()


    if total <= 0:

        weights = np.ones(
            len(outcomes)
        )

        total = weights.sum()


    probabilities = (
        weights
        /
        total
    )


    return outcomes, probabilities


# ================================================================
# 18. PREDICT ONE DELIVERY
# ================================================================

def predict_delivery(
    batter,
    bowler,
    state,
    rng
):

    expected_runs = get_runs_prediction(
        batter,
        bowler,
        state
    )


    dismissal_probability = (
        get_dismissal_probability(
            batter,
            bowler,
            state
        )
    )


    possible_runs, probabilities = (
        run_probabilities(
            expected_runs
        )
    )


    runs = int(
        rng.choice(
            possible_runs,
            p=probabilities
        )
    )


    wicket = bool(
        rng.random()
        <
        dismissal_probability
    )


    return {

        "runs":
            runs,

        "wicket":
            wicket,

        "expected_runs":
            expected_runs,

        "dismissal_probability":
            dismissal_probability
    }


# ================================================================
# 19. CHOOSE BOWLER FOR CURRENT BATTER
# ================================================================
#
# This is NOT random selection from all bowlers equally.
#
# Each available bowler gets an ML dismissal probability against
# the current batter.
#
# Stronger matchup -> greater probability of being selected.
#
# But selection remains stochastic so the same matchup doesn't
# always produce exactly the same bowling pattern.
#
# ================================================================


def choose_bowler(
    batter,
    bowling_attack,
    bowling_stats,
    state,
    rng
):

    candidates = []


    for bowler in bowling_attack:

        balls_already = bowling_stats[
            bowler
        ]["balls"]


        # Maximum 4 overs per bowler
        if balls_already >= 24:

            continue


        probability = (
            get_dismissal_probability(
                batter,
                bowler,
                state
            )
        )


        # Historical matchup run expectation
        expected_runs = (
            get_runs_prediction(
                batter,
                bowler,
                state
            )
        )


        # --------------------------------------------------------
        # Higher dismissal probability is desirable.
        # Lower expected runs is also desirable.
        #
        # Both come from ML predictions.
        # --------------------------------------------------------

        wicket_strength = (
            0.55
            +
            probability
        )


        run_control_strength = (
            1.0
            /
            (
                0.50
                +
                expected_runs
            )
        )


        score = (
            wicket_strength
            *
            run_control_strength
        )


        # Avoid zero probabilities
        score = max(
            score,
            1e-8
        )


        candidates.append(
            (
                bowler,
                score
            )
        )


    # ------------------------------------------------------------
    # If all bowlers somehow reached 4 overs, reset selection
    # cannot happen in a legal 20-over innings unless attack is
    # too small.
    # ------------------------------------------------------------

    if len(candidates) == 0:

        raise RuntimeError(
            "No legal bowler remains. "
            "Provide enough bowling options."
        )


    names = [
        item[0]
        for item in candidates
    ]


    scores = np.array(
        [
            item[1]
            for item in candidates
        ],
        dtype=float
    )


    # ------------------------------------------------------------
    # Add stochastic exploration.
    #
    # This prevents the strongest predicted bowler from being
    # selected for every single over.
    # ------------------------------------------------------------

    temperature = 0.75


    scores = np.power(
        scores,
        1.0 / temperature
    )


    probabilities = (
        scores
        /
        scores.sum()
    )


    index = rng.choice(
        len(names),
        p=probabilities
    )


    return names[index]


# ================================================================
# 20. SIMULATE ONE INNINGS
# ================================================================

def simulate_innings(
    batting_xi,
    bowling_attack,
    overs=20,
    seed=None,
    team_name="TEAM"
):

    # ------------------------------------------------------------
    # Validate
    # ------------------------------------------------------------

    batting_xi = list(
        dict.fromkeys(
            [
                str(player).strip()
                for player in batting_xi
                if str(player).strip()
            ]
        )
    )


    bowling_attack = list(
        dict.fromkeys(
            [
                str(player).strip()
                for player in bowling_attack
                if str(player).strip()
            ]
        )
    )


    if len(batting_xi) != 11:

        raise ValueError(
            "Batting XI must contain exactly 11 unique players."
        )


    if len(bowling_attack) < 5:

        raise ValueError(
            "At least 5 bowlers are required for a legal "
            "20-over innings because one bowler can bowl "
            "maximum 4 overs."
        )


    overs = int(
        overs
    )


    if overs <= 0:

        raise ValueError(
            "Overs must be greater than zero."
        )


    max_balls = (
        overs
        *
        6
    )


    # ------------------------------------------------------------
    # Seed
    # ------------------------------------------------------------

    rng = np.random.default_rng(
        seed
    )


    # ============================================================
    # MATCH STATE
    # ============================================================

    total_runs = 0

    wickets = 0

    legal_balls = 0


    # ============================================================
    # BATTING ORDER
    # ============================================================

    next_batter_index = 2


    striker = batting_xi[0]

    non_striker = batting_xi[1]


    # ============================================================
    # BATTER STATS
    # ============================================================

    batting_stats = {}


    for player in batting_xi:

        batting_stats[player] = {

            "runs": 0,

            "balls": 0,

            "fours": 0,

            "sixes": 0,

            "status": "DID NOT BAT",

            "dismissed_by": ""
        }


    batting_stats[striker][
        "status"
    ] = "NOT OUT"


    batting_stats[non_striker][
        "status"
    ] = "NOT OUT"


    # ============================================================
    # BOWLING STATS
    # ============================================================

    bowling_stats = {}


    for bowler in bowling_attack:

        bowling_stats[bowler] = {

            "balls": 0,

            "runs": 0,

            "wickets": 0
        }


    # ============================================================
    # BALL-BY-BALL LOG
    # ============================================================

    ball_log = []


    # ============================================================
    # INNINGS
    # ============================================================

    while legal_balls < max_balls:

        # --------------------------------------------------------
        # All out
        # --------------------------------------------------------

        if wickets >= 10:

            break


        # --------------------------------------------------------
        # Choose bowler
        # --------------------------------------------------------

        state = {

            "batter_runs":
                batting_stats[striker]["runs"],

            "batter_balls":
                batting_stats[striker]["balls"],

            "innings_balls":
                legal_balls,

            "wickets":
                wickets
        }


        bowler = choose_bowler(
            striker,
            bowling_attack,
            bowling_stats,
            state,
            rng
        )


        # --------------------------------------------------------
        # Predict delivery
        # --------------------------------------------------------

        outcome = predict_delivery(
            striker,
            bowler,
            state,
            rng
        )


        runs = outcome["runs"]

        wicket = outcome["wicket"]


        # --------------------------------------------------------
        # Update striker
        # --------------------------------------------------------

        batting_stats[striker][
            "runs"
        ] += runs


        batting_stats[striker][
            "balls"
        ] += 1


        if runs == 4:

            batting_stats[striker][
                "fours"
            ] += 1


        if runs == 6:

            batting_stats[striker][
                "sixes"
            ] += 1


        # --------------------------------------------------------
        # Update innings
        # --------------------------------------------------------

        total_runs += runs

        legal_balls += 1


        # --------------------------------------------------------
        # Update bowler
        # --------------------------------------------------------

        bowling_stats[bowler][
            "balls"
        ] += 1


        bowling_stats[bowler][
            "runs"
        ] += runs


        # --------------------------------------------------------
        # Current over
        # --------------------------------------------------------

        current_over = (
            (legal_balls - 1)
            // 6
        ) + 1


        ball_in_over = (
            (legal_balls - 1)
            % 6
        ) + 1


        # ========================================================
        # WICKET
        # ========================================================

        if wicket:

            wickets += 1


            batting_stats[striker][
                "status"
            ] = "OUT"


            batting_stats[striker][
                "dismissed_by"
            ] = bowler


            bowling_stats[bowler][
                "wickets"
            ] += 1


            # ----------------------------------------------------
            # Record ball
            # ----------------------------------------------------

            ball_log.append({

                "Over":
                    f"{current_over}.{ball_in_over}",

                "Batter":
                    striker,

                "Bowler":
                    bowler,

                "Runs":
                    runs,

                "Wicket":
                    "OUT",

                "Dismissed By":
                    bowler,

                "Expected Runs":
                    round(
                        outcome["expected_runs"],
                        4
                    ),

                "Dismissal Probability":
                    round(
                        outcome[
                            "dismissal_probability"
                        ],
                        4
                    )
            })


            # ----------------------------------------------------
            # If all out
            # ----------------------------------------------------

            if wickets >= 10:

                break


            # ----------------------------------------------------
            # New batter
            # ----------------------------------------------------

            if next_batter_index >= len(
                batting_xi
            ):

                break


            new_batter = batting_xi[
                next_batter_index
            ]


            next_batter_index += 1


            striker = new_batter


            batting_stats[new_batter][
                "status"
            ] = "NOT OUT"


        # ========================================================
        # NO WICKET
        # ========================================================

        else:

            ball_log.append({

                "Over":
                    f"{current_over}.{ball_in_over}",

                "Batter":
                    striker,

                "Bowler":
                    bowler,

                "Runs":
                    runs,

                "Wicket":
                    "",

                "Dismissed By":
                    "",

                "Expected Runs":
                    round(
                        outcome["expected_runs"],
                        4
                    ),

                "Dismissal Probability":
                    round(
                        outcome[
                            "dismissal_probability"
                        ],
                        4
                    )
            })


            # ----------------------------------------------------
            # Odd runs -> swap strike
            # ----------------------------------------------------

            if runs % 2 == 1:

                (
                    striker,
                    non_striker
                ) = (
                    non_striker,
                    striker
                )


        # --------------------------------------------------------
        # End of over
        # --------------------------------------------------------

        if (
            legal_balls % 6 == 0
            and
            legal_balls < max_balls
            and
            wickets < 10
        ):

            (
                striker,
                non_striker
            ) = (
                non_striker,
                striker
            )


    # ============================================================
    # FINAL STATUS
    # ============================================================

    for player in batting_xi:

        if batting_stats[player]["balls"] == 0:

            batting_stats[player][
                "status"
            ] = "DID NOT BAT"

        elif batting_stats[player][
            "status"
        ] != "OUT":

            batting_stats[player][
                "status"
            ] = "NOT OUT"


    # ============================================================
    # SCORECARD
    # ============================================================

    scorecard = []


    for number, player in enumerate(
        batting_xi,
        start=1
    ):

        item = batting_stats[player]


        runs = item["runs"]

        balls = item["balls"]


        strike_rate = (

            (
                runs
                /
                balls
                *
                100
            )

            if balls > 0

            else 0.0
        )


        scorecard.append({

            "#":
                number,

            "Batter":
                player,

            "Runs":
                runs,

            "Balls":
                balls,

            "4s":
                item["fours"],

            "6s":
                item["sixes"],

            "SR":
                round(
                    strike_rate,
                    1
                ),

            "Status":
                item["status"],

            "Dismissed By":
                item["dismissed_by"]
        })


    scorecard_df = pd.DataFrame(
        scorecard
    )


    # ============================================================
    # BOWLING SCORECARD
    # ============================================================

    bowling_card = []


    for bowler in bowling_attack:

        item = bowling_stats[bowler]


        balls = item["balls"]

        runs = item["runs"]

        wickets_taken = item["wickets"]


        economy = (

            (
                runs
                /
                balls
                *
                6
            )

            if balls > 0

            else 0.0
        )


        bowling_card.append({

            "Bowler":
                bowler,

            "Overs":
                f"{balls // 6}.{balls % 6}",

            "Balls":
                balls,

            "Runs":
                runs,

            "Wickets":
                wickets_taken,

            "Economy":
                round(
                    economy,
                    2
                )
        })


    bowling_df = pd.DataFrame(
        bowling_card
    )


    # ============================================================
    # BALL-BY-BALL DATAFRAME
    # ============================================================

    ball_log_df = pd.DataFrame(
        ball_log
    )


    # ============================================================
    # RESULT
    # ============================================================

    result = {

        "team":
            team_name,

        "runs":
            int(total_runs),

        "wickets":
            int(wickets),

        "balls":
            int(legal_balls),

        "overs":
            f"{legal_balls // 6}."
            f"{legal_balls % 6}",

        "scorecard":
            scorecard_df,

        "bowling":
            bowling_df,

        "ball_by_ball":
            ball_log_df,

        "seed":
            seed
    }


    return result


# ================================================================
# 21. MATCH SIMULATOR
# ================================================================
#
# This function takes BOTH teams dynamically.
#
# Nothing about a particular player is hard-coded here.
#
# ================================================================


def simulate_match(
    team1_batting_xi,
    team1_bowling_attack,
    team2_batting_xi,
    team2_bowling_attack,
    overs=20,
    seed=42
):

    print()
    print("=" * 80)
    print("STARTING ML MATCH SIMULATION")
    print("=" * 80)


    # ------------------------------------------------------------
    # Use different deterministic child seeds for innings.
    # ------------------------------------------------------------

    seed_sequence = np.random.SeedSequence(
        seed
    )


    child_seeds = (
        seed_sequence.spawn(2)
    )


    seed1 = int(
        child_seeds[0].generate_state(
            1
        )[0]
    )


    seed2 = int(
        child_seeds[1].generate_state(
            1
        )[0]
    )


    # ============================================================
    # TEAM 1 INNINGS
    # ============================================================

    print()
    print("=" * 80)
    print("SIMULATING TEAM 1")
    print("=" * 80)


    team1_result = simulate_innings(

        batting_xi=
            team1_batting_xi,

        bowling_attack=
            team2_bowling_attack,

        overs=
            overs,

        seed=
            seed1,

        team_name=
            "TEAM 1"
    )


    print()
    print(
        "TEAM 1:",
        f"{team1_result['runs']}/"
        f"{team1_result['wickets']}"
    )


    # ============================================================
    # TEAM 2 INNINGS
    # ============================================================

    print()
    print("=" * 80)
    print("SIMULATING TEAM 2")
    print("=" * 80)


    team2_result = simulate_innings(

        batting_xi=
            team2_batting_xi,

        bowling_attack=
            team1_bowling_attack,

        overs=
            overs,

        seed=
            seed2,

        team_name=
            "TEAM 2"
    )


    print()
    print(
        "TEAM 2:",
        f"{team2_result['runs']}/"
        f"{team2_result['wickets']}"
    )


    # ============================================================
    # WINNER
    # ============================================================

    if (
        team1_result["runs"]
        >
        team2_result["runs"]
    ):

        winner = "TEAM 1"

        margin = (
            team1_result["runs"]
            -
            team2_result["runs"]
        )

        margin_text = (
            f"{margin} runs"
        )


    elif (
        team2_result["runs"]
        >
        team1_result["runs"]
    ):

        winner = "TEAM 2"

        margin = (
            team2_result["runs"]
            -
            team1_result["runs"]
        )

        margin_text = (
            f"{margin} runs"
        )


    else:

        winner = "TIE"

        margin_text = "Tie"


    return {

        "team1":
            team1_result,

        "team2":
            team2_result,

        "winner":
            winner,

        "margin":
            margin_text,

        "seed":
            seed
    }


# ================================================================
# 22. DISPLAY SCORECARD
# ================================================================


def display_match_result(
    result
):

    team1 = result["team1"]

    team2 = result["team2"]


    print()
    print("=" * 80)
    print("TEAM 1 PREDICTED SCORECARD")
    print("=" * 80)

    display(
        team1["scorecard"]
    )


    print()
    print(
        "TOTAL:",
        f"{team1['runs']}/"
        f"{team1['wickets']}"
    )

    print(
        "OVERS:",
        team1["overs"]
    )


    print()
    print("=" * 80)
    print("TEAM 1 BOWLING CARD")
    print("=" * 80)

    display(
        team1["bowling"]
    )


    print()
    print("=" * 80)
    print("TEAM 2 PREDICTED SCORECARD")
    print("=" * 80)

    display(
        team2["scorecard"]
    )


    print()
    print(
        "TOTAL:",
        f"{team2['runs']}/"
        f"{team2['wickets']}"
    )

    print(
        "OVERS:",
        team2["overs"]
    )


    print()
    print("=" * 80)
    print("TEAM 2 BOWLING CARD")
    print("=" * 80)

    display(
        team2["bowling"]
    )


    print()
    print("=" * 80)
    print("PREDICTED MATCH RESULT")
    print("=" * 80)


    print(
        "Winner:",
        result["winner"]
    )


    print(
        "Margin:",
        result["margin"]
    )


# ================================================================
# 23. ENGINE READY
# ================================================================

print()
print("=" * 80)
print("PROPER ML BALL-BY-BALL ENGINE READY")
print("=" * 80)

print()
print("Historical matchup rows:")
print(
    len(matchup_df)
)

print()
print("Model features:")
print(
    len(MODEL_FEATURES)
)

print()
print("Ready for dynamic Custom XI prediction.")
print()

PROPER STOCHASTIC BALL-BY-BALL CUSTOM XI ML ENGINE

Current directory:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks

LOADING ML MODELS

Runs model loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_runs_model.pkl

Balls model loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_balls_model.pkl

Dismissal model loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_dismissal_model.pkl

Feature file loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_model_features.pkl

Historical matchup file loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\notebooks\matchup_df.pkl

Historical matchup rows:
61429

PROPER ML BALL-BY-BALL ENGINE READY

Historical matchup rows:
61429

Model features:
15

Ready for dynamic Custom XI prediction.



In [3]:
# ================================================================
# DYNAMIC STOCHASTIC CUSTOM XI ML MATCH SIMULATOR
# ================================================================
#
# USE THIS AFTER TRAINING YOUR CUSTOM XI MODELS.
#
# IMPORTANT:
# - No hard-coded scores
# - No hard-coded wickets
# - No hard-coded dismissals
# - No hard-coded batter performance
# - No hard-coded winner
# - No fixed seed
# - Uses batter-vs-bowler ML predictions
# - Uses stochastic sampling for different possible matches
# - Maximum 4 overs per bowler
# - Proper 20-over innings
# ================================================================

import os
import numpy as np
import pandas as pd
import joblib
import warnings

warnings.filterwarnings("ignore")

print("=" * 80)
print("DYNAMIC STOCHASTIC CUSTOM XI ML MATCH SIMULATOR")
print("=" * 80)


# ================================================================
# 1. FIND PROJECT FILES
# ================================================================

BASE_DIR = os.getcwd()

print()
print("Current directory:")
print(BASE_DIR)


def find_file(filename):

    # Search from current directory
    for root, dirs, files in os.walk(BASE_DIR):

        if filename in files:

            return os.path.join(
                root,
                filename
            )

    raise FileNotFoundError(
        f"{filename} was not found under {BASE_DIR}"
    )


# ================================================================
# 2. LOAD MODELS
# ================================================================

print()
print("=" * 80)
print("LOADING ML MODELS")
print("=" * 80)


RUNS_MODEL_PATH = find_file(
    "custom_xi_runs_model.pkl"
)

BALLS_MODEL_PATH = find_file(
    "custom_xi_balls_model.pkl"
)

DISMISSAL_MODEL_PATH = find_file(
    "custom_xi_dismissal_model.pkl"
)

FEATURES_PATH = find_file(
    "custom_xi_model_features.pkl"
)

MATCHUP_PATH = find_file(
    "matchup_df.pkl"
)


runs_model = joblib.load(
    RUNS_MODEL_PATH
)

balls_model = joblib.load(
    BALLS_MODEL_PATH
)

dismissal_model = joblib.load(
    DISMISSAL_MODEL_PATH
)

MODEL_FEATURES = joblib.load(
    FEATURES_PATH
)

matchup_df = pd.read_pickle(
    MATCHUP_PATH
)


print()
print("Runs model loaded:")
print(RUNS_MODEL_PATH)

print()
print("Balls model loaded:")
print(BALLS_MODEL_PATH)

print()
print("Dismissal model loaded:")
print(DISMISSAL_MODEL_PATH)

print()
print("Feature file loaded:")
print(FEATURES_PATH)

print()
print("Historical matchup loaded:")
print(MATCHUP_PATH)

print()
print(
    "Historical matchup rows:",
    len(matchup_df)
)


# ================================================================
# 3. CLEAN MATCHUP DATA
# ================================================================

matchup_df = matchup_df.copy()

matchup_df["batter"] = (
    matchup_df["batter"]
    .astype(str)
    .str.strip()
)

matchup_df["bowler"] = (
    matchup_df["bowler"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 4. BUILD FAST MATCHUP LOOKUP
# ================================================================

print()
print("Building matchup lookup...")


PAIR_LOOKUP = {}

for _, row in matchup_df.iterrows():

    key = (
        str(row["batter"]).strip(),
        str(row["bowler"]).strip()
    )

    PAIR_LOOKUP[key] = row


# ================================================================
# 5. BATTER HISTORY LOOKUP
# ================================================================

BATTER_HISTORY = {}

for batter, group in matchup_df.groupby("batter"):

    BATTER_HISTORY[
        str(batter).strip()
    ] = group


# ================================================================
# 6. SAFE FLOAT
# ================================================================

def safe_float(value):

    try:

        value = float(value)

        if np.isfinite(value):

            return value

    except:

        pass

    return 0.0


# ================================================================
# 7. GET BATTER-BOWLER HISTORY
# ================================================================

def get_history(
    batter,
    bowler
):

    batter = str(batter).strip()
    bowler = str(bowler).strip()

    key = (
        batter,
        bowler
    )

    # ------------------------------------------------------------
    # Direct historical matchup
    # ------------------------------------------------------------

    if key in PAIR_LOOKUP:

        row = PAIR_LOOKUP[key]

        return {

            "previous_balls":
                safe_float(
                    row.get(
                        "previous_balls",
                        0
                    )
                ),

            "previous_runs":
                safe_float(
                    row.get(
                        "previous_runs",
                        0
                    )
                ),

            "previous_dismissals":
                safe_float(
                    row.get(
                        "previous_dismissals",
                        0
                    )
                ),

            "previous_fours":
                safe_float(
                    row.get(
                        "previous_fours",
                        0
                    )
                ),

            "previous_sixes":
                safe_float(
                    row.get(
                        "previous_sixes",
                        0
                    )
                ),

            "previous_strike_rate":
                safe_float(
                    row.get(
                        "previous_strike_rate",
                        0
                    )
                ),

            "last_5_runs":
                safe_float(
                    row.get(
                        "last_5_runs",
                        0
                    )
                ),

            "last_5_balls":
                safe_float(
                    row.get(
                        "last_5_balls",
                        0
                    )
                ),

            "last_5_strike_rate":
                safe_float(
                    row.get(
                        "last_5_strike_rate",
                        0
                    )
                ),

            "last_5_dismissals":
                safe_float(
                    row.get(
                        "last_5_dismissals",
                        0
                    )
                )
        }


    # ------------------------------------------------------------
    # No direct matchup
    #
    # Use batter historical information.
    # ------------------------------------------------------------

    if batter in BATTER_HISTORY:

        rows = BATTER_HISTORY[batter]

        if len(rows) > 0:

            row = rows.iloc[-1]

            return {

                "previous_balls":
                    safe_float(
                        row.get(
                            "previous_balls",
                            0
                        )
                    ),

                "previous_runs":
                    safe_float(
                        row.get(
                            "previous_runs",
                            0
                        )
                    ),

                "previous_dismissals":
                    safe_float(
                        row.get(
                            "previous_dismissals",
                            0
                        )
                    ),

                "previous_fours":
                    safe_float(
                        row.get(
                            "previous_fours",
                            0
                        )
                    ),

                "previous_sixes":
                    safe_float(
                        row.get(
                            "previous_sixes",
                            0
                        )
                    ),

                "previous_strike_rate":
                    safe_float(
                        row.get(
                            "previous_strike_rate",
                            0
                        )
                    ),

                "last_5_runs":
                    safe_float(
                        row.get(
                            "last_5_runs",
                            0
                        )
                    ),

                "last_5_balls":
                    safe_float(
                        row.get(
                            "last_5_balls",
                            0
                        )
                    ),

                "last_5_strike_rate":
                    safe_float(
                        row.get(
                            "last_5_strike_rate",
                            0
                        )
                    ),

                "last_5_dismissals":
                    safe_float(
                        row.get(
                            "last_5_dismissals",
                            0
                        )
                    )
            }


    # ------------------------------------------------------------
    # Completely unknown combination
    # ------------------------------------------------------------

    return {

        "previous_balls": 0,
        "previous_runs": 0,
        "previous_dismissals": 0,
        "previous_fours": 0,
        "previous_sixes": 0,
        "previous_strike_rate": 0,
        "last_5_runs": 0,
        "last_5_balls": 0,
        "last_5_strike_rate": 0,
        "last_5_dismissals": 0
    }


# ================================================================
# 8. CREATE MODEL FEATURES
# ================================================================

def make_features(
    batter,
    bowler
):

    h = get_history(
        batter,
        bowler
    )


    previous_balls = h[
        "previous_balls"
    ]

    previous_runs = h[
        "previous_runs"
    ]

    previous_dismissals = h[
        "previous_dismissals"
    ]

    previous_fours = h[
        "previous_fours"
    ]

    previous_sixes = h[
        "previous_sixes"
    ]

    previous_sr = h[
        "previous_strike_rate"
    ]

    last_5_runs = h[
        "last_5_runs"
    ]

    last_5_balls = h[
        "last_5_balls"
    ]

    last_5_sr = h[
        "last_5_strike_rate"
    ]

    last_5_dismissals = h[
        "last_5_dismissals"
    ]


    previous_run_rate = (

        previous_runs /
        previous_balls

        if previous_balls > 0

        else 0
    )


    last_5_run_rate = (

        last_5_runs /
        last_5_balls

        if last_5_balls > 0

        else 0
    )


    previous_boundary_rate = (

        (
            previous_fours +
            previous_sixes
        )
        /
        previous_balls

        if previous_balls > 0

        else 0
    )


    recent_boundary_rate = (

        last_5_runs /
        last_5_balls

        if last_5_balls > 0

        else 0
    )


    dismissal_rate_history = (

        previous_dismissals /
        previous_balls

        if previous_balls > 0

        else 0
    )


    values = {

        "previous_balls":
            previous_balls,

        "previous_runs":
            previous_runs,

        "previous_dismissals":
            previous_dismissals,

        "previous_fours":
            previous_fours,

        "previous_sixes":
            previous_sixes,

        "previous_strike_rate":
            previous_sr,

        "last_5_runs":
            last_5_runs,

        "last_5_balls":
            last_5_balls,

        "last_5_strike_rate":
            last_5_sr,

        "last_5_dismissals":
            last_5_dismissals,

        "previous_run_rate":
            previous_run_rate,

        "last_5_run_rate":
            last_5_run_rate,

        "previous_boundary_rate":
            previous_boundary_rate,

        "recent_boundary_rate":
            recent_boundary_rate,

        "dismissal_rate_history":
            dismissal_rate_history
    }


    # ------------------------------------------------------------
    # IMPORTANT:
    #
    # Your saved model feature file determines the final
    # feature ordering.
    # ------------------------------------------------------------

    X = pd.DataFrame(
        [values]
    )


    # Add missing columns if necessary
    for column in MODEL_FEATURES:

        if column not in X.columns:

            X[column] = 0


    X = X[
        MODEL_FEATURES
    ]


    X = X.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )

    X = X.fillna(0)

    return X


# ================================================================
# 9. PREDICT ONE DELIVERY
# ================================================================

def predict_delivery(
    batter,
    bowler,
    rng
):

    X = make_features(
        batter,
        bowler
    )


    # ------------------------------------------------------------
    # Predict runs per ball
    # ------------------------------------------------------------

    predicted_rpb = safe_float(
        runs_model.predict(X)[0]
    )

    predicted_rpb = max(
        0.0,
        predicted_rpb
    )


    # ------------------------------------------------------------
    # Predict dismissal probability
    # ------------------------------------------------------------

    try:

        dismissal_probability = float(
            dismissal_model.predict_proba(X)[0][1]
        )

    except:

        dismissal_probability = float(
            dismissal_model.predict(X)[0]
        )


    dismissal_probability = np.clip(
        dismissal_probability,
        0.0,
        0.80
    )


    # ------------------------------------------------------------
    # RUN DISTRIBUTION
    #
    # ML predicted expected runs controls the probability.
    #
    # The result is stochastic, so two predictions need not be
    # identical.
    # ------------------------------------------------------------

    possible_runs = np.array(
        [
            0,
            1,
            2,
            3,
            4,
            6
        ],
        dtype=int
    )


    # Spread adapts to expected run rate
    spread = max(
        0.65,
        min(
            1.50,
            0.75 + predicted_rpb * 0.25
        )
    )


    distances = np.abs(
        possible_runs -
        predicted_rpb
    )


    weights = np.exp(
        -distances /
        spread
    )


    # ------------------------------------------------------------
    # Make higher outcomes more likely when ML predicts
    # aggressive batting.
    # ------------------------------------------------------------

    weights = weights.astype(float)


    if predicted_rpb >= 0.90:

        weights[
            possible_runs == 4
        ] *= 1.25

        weights[
            possible_runs == 6
        ] *= 1.10


    elif predicted_rpb <= 0.35:

        weights[
            possible_runs == 0
        ] *= 1.25

        weights[
            possible_runs == 1
        ] *= 1.10


    weights = (
        weights /
        weights.sum()
    )


    runs = int(
        rng.choice(
            possible_runs,
            p=weights
        )
    )


    # ------------------------------------------------------------
    # WICKET
    # ------------------------------------------------------------

    wicket = (
        rng.random()
        <
        dismissal_probability
    )


    return {

        "runs":
            runs,

        "wicket":
            bool(wicket),

        "dismissal_probability":
            dismissal_probability,

        "predicted_rpb":
            predicted_rpb
    }


# ================================================================
# 10. CALCULATE BOWLER STRENGTH
# ================================================================

def bowler_strength(
    batter,
    bowler
):

    X = make_features(
        batter,
        bowler
    )


    try:

        probability = float(
            dismissal_model.predict_proba(X)[0][1]
        )

    except:

        probability = float(
            dismissal_model.predict(X)[0]
        )


    return max(
        0.0001,
        probability
    )


# ================================================================
# 11. CHOOSE BOWLER
# ================================================================
#
# IMPORTANT:
#
# We do NOT choose randomly from unlimited bowlers.
#
# A bowler:
#   maximum = 4 overs
#
# ML matchup strength influences selection.
#
# A bowler who is particularly strong against the current
# striker receives greater probability.
# ================================================================

def choose_bowler(
    striker,
    bowling_attack,
    bowling_stats,
    over_number,
    rng
):

    available = []


    for bowler in bowling_attack:

        bowler_balls = bowling_stats[
            bowler
        ]["balls"]

        # Maximum 4 overs = 24 balls
        if bowler_balls < 24:

            available.append(
                bowler
            )


    if not available:

        raise RuntimeError(
            "No bowler available. "
            "Check bowling attack configuration."
        )


    # ------------------------------------------------------------
    # Prefer bowlers who have not bowled previous over
    # ------------------------------------------------------------

    preferred = [

        bowler

        for bowler in available

        if bowling_stats[
            bowler
        ]["last_over"] != over_number - 1
    ]


    if preferred:

        available = preferred


    # ------------------------------------------------------------
    # ML dismissal strength
    # ------------------------------------------------------------

    scores = []


    for bowler in available:

        strength = bowler_strength(
            striker,
            bowler
        )


        balls_bowled = bowling_stats[
            bowler
        ]["balls"]


        # Slight workload preference for bowlers with fewer balls.
        workload_factor = 1.0 / (
            1.0 +
            balls_bowled * 0.035
        )


        score = (
            strength *
            workload_factor
        )


        scores.append(
            max(
                score,
                0.0001
            )
        )


    scores = np.array(
        scores,
        dtype=float
    )


    scores = (
        scores /
        scores.sum()
    )


    index = rng.choice(
        len(available),
        p=scores
    )


    return available[index]


# ================================================================
# 12. SWAP STRIKE
# ================================================================

def swap_strike(
    striker,
    non_striker
):

    return (
        non_striker,
        striker
    )


# ================================================================
# 13. SIMULATE ONE INNINGS
# ================================================================

def simulate_innings(
    batting_xi,
    bowling_attack,
    team_name="TEAM",
    overs=20,
    rng=None
):

    if rng is None:

        rng = np.random.default_rng()


    # ------------------------------------------------------------
    # Remove duplicates while preserving order
    # ------------------------------------------------------------

    batting_xi = list(
        dict.fromkeys(
            batting_xi
        )
    )

    bowling_attack = list(
        dict.fromkeys(
            bowling_attack
        )
    )


    if len(batting_xi) != 11:

        raise ValueError(
            f"{team_name}: batting XI must contain exactly 11 "
            f"players. Got {len(batting_xi)}."
        )


    if len(bowling_attack) < 5:

        raise ValueError(
            f"{team_name}: at least 5 bowlers are required "
            f"for a 20-over innings."
        )


    # ============================================================
    # MATCH STATE
    # ============================================================

    total_runs = 0

    wickets = 0

    legal_balls = 0

    max_balls = (
        overs *
        6
    )


    # ============================================================
    # BATTERS
    # ============================================================

    striker_index = 0

    non_striker_index = 1

    next_batter_index = 2


    striker = batting_xi[
        striker_index
    ]

    non_striker = batting_xi[
        non_striker_index
    ]


    # ============================================================
    # BATTER STATISTICS
    # ============================================================

    stats = {}


    for player in batting_xi:

        stats[player] = {

            "runs": 0,

            "balls": 0,

            "fours": 0,

            "sixes": 0,

            "status": "DID NOT BAT",

            "dismissed_by": ""
        }


    stats[striker][
        "status"
    ] = "NOT OUT"


    stats[non_striker][
        "status"
    ] = "NOT OUT"


    # ============================================================
    # BOWLER STATISTICS
    # ============================================================

    bowling_stats = {}


    for bowler in bowling_attack:

        bowling_stats[bowler] = {

            "balls": 0,

            "runs": 0,

            "wickets": 0,

            "last_over": -99
        }


    # ============================================================
    # BALL-BY-BALL SIMULATION
    # ============================================================

    while legal_balls < max_balls:

        # --------------------------------------------------------
        # ALL OUT
        # --------------------------------------------------------

        if wickets >= 10:

            break


        # --------------------------------------------------------
        # Need another batter
        # --------------------------------------------------------

        if next_batter_index >= 11 and (
            stats[striker]["status"] == "OUT"
            or
            stats[non_striker]["status"] == "OUT"
        ):

            break


        # --------------------------------------------------------
        # Current over
        # --------------------------------------------------------

        over_number = (
            legal_balls // 6
        )


        # --------------------------------------------------------
        # Select bowler
        # --------------------------------------------------------

        bowler = choose_bowler(

            striker,

            bowling_attack,

            bowling_stats,

            over_number,

            rng
        )


        # --------------------------------------------------------
        # Predict delivery
        # --------------------------------------------------------

        outcome = predict_delivery(

            striker,

            bowler,

            rng
        )


        runs = int(
            outcome["runs"]
        )

        wicket = bool(
            outcome["wicket"]
        )


        # --------------------------------------------------------
        # Update batter
        # --------------------------------------------------------

        stats[striker][
            "runs"
        ] += runs


        stats[striker][
            "balls"
        ] += 1


        if runs == 4:

            stats[striker][
                "fours"
            ] += 1


        if runs == 6:

            stats[striker][
                "sixes"
            ] += 1


        # --------------------------------------------------------
        # Match totals
        # --------------------------------------------------------

        total_runs += runs

        legal_balls += 1


        # --------------------------------------------------------
        # Bowler totals
        # --------------------------------------------------------

        bowling_stats[
            bowler
        ]["balls"] += 1


        bowling_stats[
            bowler
        ]["runs"] += runs


        # ========================================================
        # WICKET
        # ========================================================

        if wicket:

            wickets += 1


            stats[striker][
                "status"
            ] = "OUT"


            stats[striker][
                "dismissed_by"
            ] = bowler


            bowling_stats[
                bowler
            ]["wickets"] += 1


            # ----------------------------------------------------
            # New batter enters
            # ----------------------------------------------------

            if next_batter_index < 11:

                new_batter = batting_xi[
                    next_batter_index
                ]

                next_batter_index += 1


                stats[new_batter][
                    "status"
                ] = "NOT OUT"


                striker = new_batter


                striker_index = (
                    batting_xi.index(
                        striker
                    )
                )


            else:

                break


        else:

            # ----------------------------------------------------
            # Change strike for odd runs
            # ----------------------------------------------------

            if runs % 2 == 1:

                (
                    striker,
                    non_striker
                ) = swap_strike(
                    striker,
                    non_striker
                )


        # ========================================================
        # END OF OVER
        # ========================================================

        if legal_balls % 6 == 0:

            bowling_stats[
                bowler
            ]["last_over"] = over_number


            (
                striker,
                non_striker
            ) = swap_strike(
                striker,
                non_striker
            )


    # ================================================================
    # FINAL BATTER STATUS
    # ================================================================

    for player in batting_xi:

        if stats[player]["balls"] > 0:

            if stats[player]["status"] != "OUT":

                stats[player][
                    "status"
                ] = "NOT OUT"

        else:

            stats[player][
                "status"
            ] = "DID NOT BAT"


    # ================================================================
    # BATTING SCORECARD
    # ================================================================

    scorecard = []


    for position, player in enumerate(
        batting_xi,
        start=1
    ):

        item = stats[player]


        runs = item[
            "runs"
        ]

        balls = item[
            "balls"
        ]


        strike_rate = (

            (
                runs /
                balls
            ) *
            100

            if balls > 0

            else 0
        )


        scorecard.append({

            "No":
                position,

            "Batter":
                player,

            "Runs":
                runs,

            "Balls":
                balls,

            "4s":
                item["fours"],

            "6s":
                item["sixes"],

            "SR":
                round(
                    strike_rate,
                    1
                ),

            "Status":
                item["status"],

            "Dismissed By":
                item["dismissed_by"]
        })


    scorecard_df = pd.DataFrame(
        scorecard
    )


    # ================================================================
    # BOWLING SCORECARD
    # ================================================================

    bowling_card = []


    for bowler in bowling_attack:

        item = bowling_stats[
            bowler
        ]


        balls = item[
            "balls"
        ]


        economy = (

            (
                item["runs"] /
                balls
            ) *
            6

            if balls > 0

            else 0
        )


        bowling_card.append({

            "Bowler":
                bowler,

            "Overs":
                f"{balls // 6}."
                f"{balls % 6}",

            "Runs":
                item["runs"],

            "Wickets":
                item["wickets"],

            "Economy":
                round(
                    economy,
                    2
                )
        })


    bowling_df = pd.DataFrame(
        bowling_card
    )


    # ================================================================
    # RESULT
    # ================================================================

    return {

        "team":
            team_name,

        "runs":
            int(total_runs),

        "wickets":
            int(wickets),

        "balls":
            int(legal_balls),

        "overs":
            f"{legal_balls // 6}."
            f"{legal_balls % 6}",

        "scorecard":
            scorecard_df,

        "bowling":
            bowling_df
    }


# ================================================================
# 14. MATCH SIMULATION
# ================================================================

def simulate_match(
    team1_batting,
    team1_bowling,
    team2_batting,
    team2_bowling,
    overs=20
):

    # ------------------------------------------------------------
    # NO FIXED SEED
    #
    # Every call gets fresh randomness.
    # ------------------------------------------------------------

    rng = np.random.default_rng()


    print()
    print("=" * 80)
    print("SIMULATING TEAM 1 INNINGS")
    print("=" * 80)


    team1_result = simulate_innings(

        batting_xi=team1_batting,

        bowling_attack=team2_bowling,

        team_name="TEAM 1",

        overs=overs,

        rng=rng
    )


    print()
    print(
        "TEAM 1:",
        f"{team1_result['runs']}/"
        f"{team1_result['wickets']}"
        f" ({team1_result['overs']})"
    )


    print()
    print("=" * 80)
    print("SIMULATING TEAM 2 INNINGS")
    print("=" * 80)


    team2_result = simulate_innings(

        batting_xi=team2_batting,

        bowling_attack=team1_bowling,

        team_name="TEAM 2",

        overs=overs,

        rng=rng
    )


    print()
    print(
        "TEAM 2:",
        f"{team2_result['runs']}/"
        f"{team2_result['wickets']}"
        f" ({team2_result['overs']})"
    )


    # ============================================================
    # WINNER
    # ============================================================

    if team1_result["runs"] > team2_result["runs"]:

        winner = "TEAM 1"

        margin = (
            team1_result["runs"] -
            team2_result["runs"]
        )

        result_text = (
            f"TEAM 1 won by "
            f"{margin} runs"
        )


    elif team2_result["runs"] > team1_result["runs"]:

        winner = "TEAM 2"

        margin = (
            team2_result["runs"] -
            team1_result["runs"]
        )

        result_text = (
            f"TEAM 2 won by "
            f"{margin} runs"
        )


    else:

        winner = "TIE"

        result_text = "Match tied"


    return {

        "team1":
            team1_result,

        "team2":
            team2_result,

        "winner":
            winner,

        "result":
            result_text
    }


# ================================================================
# 15. READY
# ================================================================

print()
print("=" * 80)
print("CUSTOM XI ML ENGINE READY")
print("=" * 80)

print()
print("Features:")
print("✓ Batter vs bowler historical matchup")
print("✓ ML runs prediction")
print("✓ ML dismissal prediction")
print("✓ Stochastic delivery outcomes")
print("✓ Dynamic bowler selection")
print("✓ Maximum 4 overs per bowler")
print("✓ Dynamic wickets")
print("✓ Dynamic dismissals")
print("✓ Dynamic batting scores")
print("✓ Dynamic bowling figures")
print("✓ No fixed random seed")
print("✓ No hard-coded match result")

print()
print("Ready for dynamic Custom XI prediction.")

DYNAMIC STOCHASTIC CUSTOM XI ML MATCH SIMULATOR

Current directory:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks

LOADING ML MODELS

Runs model loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_runs_model.pkl

Balls model loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_balls_model.pkl

Dismissal model loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_dismissal_model.pkl

Feature file loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\custom_xi_model_features.pkl

Historical matchup loaded:
c:\Users\SAI PAVAN KARTHIK\OneDrive\Desktop\Versus\notebooks\notebooks\matchup_df.pkl

Historical matchup rows: 61429

Building matchup lookup...

CUSTOM XI ML ENGINE READY

Features:
✓ Batter vs bowler historical matchup
✓ ML runs prediction
✓ ML dismissal prediction
✓ Stochastic delivery outcomes
✓ Dynamic bowler selection
✓ Maximum 4 overs per bowler
✓ Dynamic wickets
✓ Dynami

In [6]:
# ================================================================
# CUSTOM XI MATCH TEST
# ================================================================

TEAM_1_BATTING = [
    "RG Sharma",
    "V Kohli",
    "Shubman Gill",
    "SS Iyer",
    "KL Rahul",
    "HH Pandya",
    "MS Dhoni",
    "AR Patel",
    "JJ Bumrah",
    "Kuldeep Yadav",
    "B Kumar"
]

TEAM_1_BOWLING = [
    "JJ Bumrah",
    "B Kumar",
    "HH Pandya",
    "AR Patel",
    "Kuldeep Yadav"
]


TEAM_2_BATTING = [
    "TM Head",
    "JC Buttler",
    "H Klaasen",
    "KA Pollard",
    "GJ Maxwell",
    "SO Hetmyer",
    "PJ Cummins",
    "MA Starc",
    "A Zampa",
    "K Rabada",
    "L Ngidi"
]

TEAM_2_BOWLING = [
    "PJ Cummins",
    "MA Starc",
    "A Zampa",
    "K Rabada",
    "GJ Maxwell"
]


# ================================================================
# RUN MATCH
# ================================================================

match = simulate_match(
    team1_batting=TEAM_1_BATTING,
    team1_bowling=TEAM_1_BOWLING,
    team2_batting=TEAM_2_BATTING,
    team2_bowling=TEAM_2_BOWLING,
    overs=20
)


# ================================================================
# TEAM 1
# ================================================================

print()
print("=" * 80)
print("TEAM 1 SCORECARD")
print("=" * 80)

display(
    match["team1"]["scorecard"]
)

print()

print(
    "TEAM 1 TOTAL:",
    f"{match['team1']['runs']}/"
    f"{match['team1']['wickets']}"
    f" ({match['team1']['overs']})"
)

print()
print("TEAM 2 BOWLING")

display(
    match["team2"]["bowling"]
)


# ================================================================
# TEAM 2
# ================================================================

print()
print("=" * 80)
print("TEAM 2 SCORECARD")
print("=" * 80)

display(
    match["team2"]["scorecard"]
)

print()

print(
    "TEAM 2 TOTAL:",
    f"{match['team2']['runs']}/"
    f"{match['team2']['wickets']}"
    f" ({match['team2']['overs']})"
)

print()
print("TEAM 1 BOWLING")

display(
    match["team1"]["bowling"]
)


# ================================================================
# RESULT
# ================================================================

print()
print("=" * 80)
print("PREDICTED RESULT")
print("=" * 80)

print(
    match["result"]
)


SIMULATING TEAM 1 INNINGS

TEAM 1: 124/10 (14.1)

SIMULATING TEAM 2 INNINGS

TEAM 2: 202/7 (20.0)

TEAM 1 SCORECARD


,No,Batter,Runs,Balls,4s,6s,SR,Status,Dismissed By
0,1,RG Sharma,1,1,0,0,100.0,OUT,GJ Maxwell
1,2,V Kohli,2,2,0,0,100.0,OUT,MA Starc
2,3,Shubman Gill,4,3,0,0,133.3,OUT,MA Starc
3,4,SS Iyer,6,2,1,0,300.0,OUT,GJ Maxwell
4,5,KL Rahul,30,18,0,1,166.7,OUT,K Rabada
5,6,HH Pandya,18,11,1,0,163.6,OUT,MA Starc
6,7,MS Dhoni,16,11,2,0,145.5,OUT,K Rabada
7,8,AR Patel,14,12,0,0,116.7,OUT,GJ Maxwell
8,9,JJ Bumrah,22,15,0,0,146.7,OUT,K Rabada
9,10,Kuldeep Yadav,3,3,0,0,100.0,OUT,MA Starc



TEAM 1 TOTAL: 124/10 (14.1)

TEAM 2 BOWLING


,Bowler,Overs,Runs,Wickets,Economy
0,JJ Bumrah,4.0,35,0,8.75
1,B Kumar,4.0,37,2,9.25
2,HH Pandya,4.0,46,0,11.50
3,AR Patel,4.0,41,2,10.25
4,Kuldeep Yadav,4.0,43,3,10.75



TEAM 2 SCORECARD


,No,Batter,Runs,Balls,4s,6s,SR,Status,Dismissed By
0,1,TM Head,36,26,0,0,138.5,OUT,Kuldeep Yadav
1,2,JC Buttler,56,27,4,2,207.4,OUT,Kuldeep Yadav
2,3,H Klaasen,9,7,0,0,128.6,OUT,B Kumar
3,4,KA Pollard,59,33,1,0,178.8,OUT,AR Patel
4,5,GJ Maxwell,1,1,0,0,100.0,OUT,Kuldeep Yadav
5,6,SO Hetmyer,25,14,1,0,178.6,OUT,B Kumar
6,7,PJ Cummins,2,2,0,0,100.0,OUT,AR Patel
7,8,MA Starc,10,8,0,0,125.0,NOT OUT,
8,9,A Zampa,4,2,0,0,200.0,NOT OUT,
9,10,K Rabada,0,0,0,0,0.0,DID NOT BAT,



TEAM 2 TOTAL: 202/7 (20.0)

TEAM 1 BOWLING


,Bowler,Overs,Runs,Wickets,Economy
0,PJ Cummins,2.0,15,0,7.50
1,MA Starc,3.5,44,4,11.48
2,A Zampa,2.2,16,0,6.86
3,K Rabada,2.1,19,3,8.77
4,GJ Maxwell,3.5,30,3,7.83



PREDICTED RESULT
TEAM 2 won by 78 runs
